<div style="background: linear-gradient(135deg, #e9f2fb 0%, #f7fbff 100%); padding: 18px 20px; border: 1px solid #d5e3f0; border-radius: 10px; text-align: center;">
  <div style="font-size: 32px; font-weight: 700; color: #1f3b57; letter-spacing: 0.2px;">NX-414 Brain-like Computation and Intelligence</div>
</div>


<div style="text-align: center; font-size: 21px; font-weight: 600; color: #36536b; margin-top: 6px;">Project Notebook — Spring 2026</div>


<div style="text-align: center; color: #4f6478; font-size: 16px; margin-bottom: 10px;">Brain–Model Alignment Across Neural Recording Modalities</div>
<div style="text-align: center; color: #6b7280; font-size: 13px;">Prepared by: Abdulkadir Gokce</div>

---


# Group Information

Fill in this section at the top of your notebook and report.

- **Group member 1:** Lucien Zimmermann,346884, lucien.zimmermann@epfl.ch  
- **Group member 2:** Thibault Schlegel, 345787, thibault.schlegel@epfl.ch  
- **Group member 3:** Gauthier Huguelet, 329078, gauthier.huguelet@epfl.ch  

---

# What You Must Submit

Submit the following files:

1. **One Jupyter notebook** containing your full analysis.
2. **Any supporting Python scripts** needed to run the notebook.
3. **Figures that are part of your notebook answers** should be embedded and rendered in notebook Markdown.
4. **One PDF report** of **up to 2 pages**, **excluding references**, with **no appendix**.
5. **One zip archive** named exactly as:

```text
nx414_{SCIPER1}_{SCIPER2}_{SCIPER3}.zip
```

If your group has fewer than three members, reduce the number of `_SCIPER` fields accordingly.

## Submission Rules

- **Clear all notebook outputs before submission.**
- If outputs are not cleared, we will clear them ourselves and grade the cleaned notebook.
- Submit only the code required to reproduce your results.
- **Do not submit model weights.**
- **Do not submit CSV files or other large derived result dumps.**
- Keep the archive lightweight and reproducible.
- For the **final notebook**, any figure you want to present as part of your scientific argument should be **embedded in Markdown with accompanying interpretation**, rather than left as a raw cell output with no explanation.

Failure to follow these instructions may reduce your final grade.

## Use of LLMs

You may use LLM-based tools to help you write code, debug, or improve explanations. However, you remain fully responsible for the **correctness**, **quality**, and **clarity** of everything you submit, including both the notebook and the report.

In particular:

- check that any generated code actually runs and does what you claim it does,
- verify that any scientific statement or interpretation is correct,
- make sure the final writing sounds like a clear academic report written for this course,
- avoid vague, overly polished, or context-free text,
- avoid fancy wording or unnecessarily complex sentences that do not add clarity.

If you use an LLM, revise the output so that your submission reads naturally, is specific to your actual results, and does not look like generic generated text.

Failure to do so may result in **a point deduction**.

## Expected scope

Because this project spans roughly **three weeks** and counts for **30% of the final course grade**, the expected output is closer to a **compact course project** than to a one-week homework notebook. Your submission should therefore read like a small empirical study: it should be clearly structured, contain short written interpretations throughout, compare alternatives systematically, and end with a coherent synthesis of your main findings.

A strong notebook will not only run end-to-end, but will also explain **why** each analysis is being performed, what each metric is meant to capture, and what the results imply about the strengths and limitations of the models and datasets.

At the same time, **some parts of the project are intentionally left a bit loose**. This is by design: beyond implementing the required core analyses, you are expected to make reasonable scientific choices, justify them clearly, and show some ingenuity in how you explore the data and compare models.

## Suggested 3-week pacing

Use the notebook structure below to organize your work over the three weeks.

- **Week 1:** complete **Section 0** and **Section 1**. Understand the datasets, verify stimulus matching, inspect the processed responses, and start the visualization and reliability analyses.
- **Week 2:** complete the required analyses in **Section 2**. In this section, you must complete both the representational and predictive parts of the project. Begin **Section 3** by brainstorming possible extensions and sketching out a plan for your chosen extension.
- **Week 3:** complete **Section 3**, polish the notebook, select the strongest figures, and write the 2-page report.


---

# 0. Introduction and Setup

## 0.1 Project goal

In this project, you will study how neural responses from different recording modalities align with features extracted from two vision models. More specifically, you will work in the standard **brain–model alignment** setting: a model processes an image, a candidate internal layer is selected, and that representation is compared to measured neural responses using representational and predictive metrics.

The notebook is organized around four sections:

- **Section 0:** introduction, setup, and understanding the provided resources.
- **Section 1:** dataset inspection, visualization, and noise ceiling estimation.
- **Section 2:** brain–model alignment through both representational metrics and predictive linear models.
- **Section 3:** an open-ended extension beyond the baseline pipeline.

## 0.2 Why task-optimized models?

Task-optimized neural networks are among the most useful current **in-silico models of sensory cortex**. The central idea is simple: instead of hand-designing a model to mimic every biological detail, we optimize a model to perform a meaningful visual task and then ask whether its internal representations resemble those found in the brain. This approach has been highly influential because models trained to solve vision tasks often develop representations that predict activity along the visual hierarchy surprisingly well.

These models are useful scientifically because they provide **testable computational hypotheses**. If a model layer predicts neural responses well, that does not mean the brain literally implements the same mechanism, but it does suggest that the layer may encode information in a similar format or at a similar level of abstraction. Brain–model alignment is therefore a way to ask not just whether a model is accurate on a task, but whether it organizes visual information in a brain-relevant way.

## 0.3 Why compare multiple modalities?

A single recording modality gives only a partial view of neural computation. In this project, you will work with **electrophysiology, EEG, and fMRI**, which differ in temporal resolution, spatial resolution, and what exactly is measured. Looking across modalities helps you see which conclusions are robust and which depend on the measurement scale.

## 0.4 Learning goals

By the end of this project, you should be able to:

- inspect and summarize neural datasets from multiple modalities,
- visualize neural signals and data quality,
- implement and compare **two noise ceiling estimators**,
- implement **RSA** and **unbiased linear CKA**,
- fit **linear encoding models** from model features to neural responses,
- compare alignment across **models, layers, ROIs, and metrics**,
- interpret what each alignment metric captures,
- design and evaluate one meaningful extension beyond the baseline pipeline.

A strong submission should therefore demonstrate both **technical correctness** and **scientific reasoning**: beyond obtaining scores, you should be able to explain why a dataset is noisy, why one layer may outperform another, and why representational and predictive metrics sometimes disagree.

## 0.5 Provided data

All main data are stored in `/shared/NX-414/data/`.

### Background: processed data derivatives

The files provided for this project are **not raw neural recordings**. They are already processed, analysis-ready derivatives produced with modality-appropriate pipelines. This is important scientifically: many of your results will depend not only on the model features, but also on preprocessing choices such as repetition averaging, denoising, response-window selection, voxel/channel filtering, and how reliability is estimated.

We performed the preprocessing for you because these pipelines often require substantial modality-specific expertise, time, and compute.

At a high level, the datasets used here were prepared as follows:

- **TVSD (macaque electrophysiology)** — normalized multi-unit responses from ventral-stream areas. Responses were z-scored within session, firing rates were averaged in an analysis window centered on each site’s response peak, low-reliability channels were excluded, and repeated test responses were averaged for evaluation.
- **THINGS-EEG2 (human EEG)** — source EEG responses resampled to **100 Hz**. Noise ceilings were computed per subject, channel, and time point, and repetitions were averaged within train and test splits.
- **NSD (human fMRI)** — **b3 single-trial beta estimates** in `func1pt8mm` space, derived using voxel-wise HRF fitting, GLMdenoise, and ridge regression. Analyses are restricted to ROI-defined visually responsive voxels, low-reliability voxels are filtered out, and responses are averaged across available repetitions.

You are **not** expected to re-run the full preprocessing pipelines. You **are** expected to understand what kinds of neural quantities you are analyzing, what has already been averaged or denoised, and how these choices affect interpretation.

### Main neural datasets

- **`tvsd.h5`** — macaque electrophysiology from **2 monkeys**, with **22,248 train** and **100 test** stimuli, covering **V1, V4, and IT**.
- **`things_eeg2.h5`** — human EEG from **10 subjects**, with **16,540 train** and **200 test** stimuli, with region groupings such as **occipital**, **parietal**, **temporal**, **frontal**, **central**, **occipital_parietal**, and **whole_brain**.
- **`nsd_func1pt8mm_individualROIs.h5`** — human fMRI from **8 subjects**, with roughly **9,000 train** and **1,000 test** stimuli per subject, across multiple visual ROIs.

### Additional files

- `things_eeg2-test_reps.h5`  
  EEG test responses **with repetitions and without averaging**.  
  Use this file to implement and compare **two noise ceiling estimators**.

- `nsd-subj01-ncsnr-{lh,rh}.mgh`  
  Surface-based NSD reliability values for `subj-01` on **fsaverage**.  
  Use these to visualize cortical reliability and convert **ncsnr** into **noise ceiling**.

### Neural response shapes

- **TVSD:** `(n_stimuli, n_units)`
- **EEG2:** `(n_stimuli, n_channels, n_timepoints)`
- **NSD:** `(n_stimuli, n_voxels)`

For EEG, the time axis contains **80 time points** sampled at **100 Hz**, covering **0.0 s to 0.8 s**.

### Noise ceilings

Noise ceilings are stored per target:

- per neuron for **TVSD**,
- per channel × time point for **EEG2**,
- per voxel for **NSD**.

They are stored as **percent reliability**.  
To convert them to the range `[0, 1]`, divide by `100`.

In this project, the provided noise ceilings are mainly intended for **predictive metrics** such as **Pearson correlation** and **explained variance**. They reflect the reliability of the neural responses and therefore define an upper bound on how well any model can predict those responses.

When you compute predictive metrics, you should apply a noise ceiling correction to account for this upper bound. The standard idea is simple: divide the raw predictive score by the corresponding noise ceiling value for that target.

The provided noise ceilings are defined for **explained variance**. If you want to apply the same logic to **Pearson correlation**, first convert the explained-variance ceiling into a correlation ceiling by taking the element-wise square root, and then divide the raw correlation by that quantity. For a more detailed discussion of noise ceiling correction, see van Bree et al. (2025).

You may therefore report both **raw** and **noise-ceiling-corrected** predictive scores where appropriate.


For example, if the provided ceiling is an explained-variance reliability estimate, you can compute a noise-corrected Pearson correlation as:

```python
r_nc = r / np.sqrt(ev_ceiling)
```

where `r` is the raw Pearson correlation and `ev_ceiling` is the explained-variance ceiling expressed on the range `[0, 1]`.

By contrast, **RSA** and **CKA** should typically be reported as **raw values** in this project. Noise ceilings for representational similarity metrics require a different methodology and are **not** provided here.


Do **not** apply this correction directly to **RSA** or **CKA**.

### Stimulus identifiers

- **TVSD / EEG2:** byte strings such as `b'aardvark/aardvark_01b.jpg'`
- **NSD:** integer stimulus IDs

## 0.6 Model features

All extracted features are stored in `/shared/NX-414/extracted_features/`.

The feature files contain **internal activations extracted from multiple candidate layers** while the models process the same images shown in the neural experiments. You can think of each layer as a representation matrix of shape roughly **stimuli × features**. These layer-wise representations are what you will compare to the brain using RSA, CKA, and encoding models.

Feature extractions across models were made tractable by projecting activations to **30,000 dimensions** using a random projection. The provided feature files follow that same idea. In practice, this means you can treat the feature vectors as compact surrogates for the original activations while still performing meaningful alignment analyses.

### Model A: `adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0`

- Architecture: **ResNet-152**
- Pretraining: ImageNet + adversarial fine-tuning
- Available feature files:
  - `things_stimuli.h5`
  - `nsd_stimuli.h5`

### Model B: `Qwen3-VL-2B-Instruct`

- Architecture: **vision-language transformer**
- Available feature files:
  - `things_stimuli.h5`
  - `nsd_stimuli.h5`

For both models, layers have been projected to **30,000 dimensions** using random projections.

### Feature extraction note

For this project, feature extraction has already been done for you. Your job is therefore not to run the vision models on raw images, but to understand **which layer** to use, how to match feature rows to neural stimuli, and what different layers reveal about representational hierarchy.

### Important note for NSD

For NSD, the feature files contain features for **all 73,000 images**, but each subject saw only a subset (~9,000).  
You must therefore select feature rows using the subject-specific NSD stimulus IDs.

## 0.7 Matching features to neural responses

You must match neural responses and features through the stimulus IDs.

### THINGS-based datasets: TVSD and EEG2

For these datasets, both neural IDs and feature IDs are byte strings. Matching should therefore be exact.

### NSD

For NSD, both sides use integer IDs. The feature file contains all 73,000 stimuli, while each subject has only a subset. Use the subject-specific NSD stimulus IDs to select the corresponding feature rows.

### Recommended procedure

```python
feat_ids = feature_file['ids'][:]
id_to_feat_idx = {id_: i for i, id_ in enumerate(feat_ids)}
feat_idx = np.array([id_to_feat_idx[x] for x in neural_ids])
```

To make HDF5 reads efficient:

1. build the index array,
2. sort indices before reading,
3. load only the required rows,
4. restore the original order.

## 0.8 General analysis rules

### Train/test discipline

Do **not** use the test split for model selection or hyperparameter tuning. Use a validation split or cross-validation within the training data.

### EEG targets

For EEG, you may either:

- flatten the targets to `(n_stimuli, n_channels * n_timepoints)`, or
- fit separate models per channel × time point.

Either choice is acceptable, but you must explain your decision clearly.

### Predictive metrics

Report the following predictive metrics where appropriate:

- `pearsonr`
- `pearsonr_nc`
- `explained_variance`
- `explained_variance_nc`

The provided EEG signals are not filtered like the other datasets. As a result, some low-reliability channels or time points can produce unstable predictive scores. When averaging predictive metrics over EEG targets, apply an on-the-fly filter such as **noise ceiling < 0.1**.

### Representational metrics

Also report:

- `RSA`
- `CKA`
- `encoding-RSA`
- `encoding-CKA`

Encoding RSA/CKA is a hybrid metric where you compute RSA or CKA between the predicted neural responses (from the linear encoding model) and the actual neural responses, instead of just comparing model activation directly. This can help you understand whether the linear encoding model captures the representational geometry of the neural data, beyond just predicting individual response values. You can refer to Conwell et al. (2024) for more details on this approach. 

Use only the test split for computing representational metrics.

Do **not** noise-correct RSA or CKA using the predictive-metric procedure.

## 0.9 Setup and data loading

Your notebook should begin with a short introduction, clear imports, utility functions, and a brief verification that the provided files are correctly organized and matched.

**Section 0 is required but not graded separately.** It is treated as setup for the rest of the project. Missing or incorrect setup may reduce scores in later sections if it affects correctness or reproducibility.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Import the required packages.
- Define utility functions.
- Load metadata for all datasets.
- Inspect the structure of each `.h5` file.
- Load feature metadata for both models.
- Verify that feature IDs and neural stimulus IDs match.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Dataset and feature overview:** one compact table or printed summary covering all neural datasets and both feature sets.
2. **Stimulus-matching verification:** explicit checks or assertions showing that stimulus matching works for THINGS-based datasets and for NSD.
3. **Short structural summary:** a short written note describing the main structural differences across datasets.

<div style="background:#eef8f4; border-left:4px solid #5b9a7a; padding:8px 12px; border-radius:6px; font-weight:700; color:#285943;">Questions you should answer</div>

- How many stimuli are available in each dataset?
- What is the shape of the neural response tensor in each dataset?
- Which datasets contain subjects, ROIs, repetitions, channels, or time points?
- What are the feature dimensionalities across layers?

In [1]:
# TODO: imports
# TODO: define paths
# TODO: inspect dataset structure
# TODO: verify stimulus matching

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn import metrics
from scipy import stats
import nibabel as nib
from nilearn import plotting as nlplt
import h5py
import torch
from pathlib import Path

# Paths

DATA_DIR  = Path("/shared/NX-414/data")
FEAT_DIR  = Path("/shared/NX-414/extracted_features")

# Neural datasets
TVSD_PATH      = DATA_DIR / "tvsd.h5"
EEG_PATH       = DATA_DIR / "things_eeg2.h5"
EEG_REPS_PATH  = DATA_DIR / "things_eeg2-test_reps.h5"
NSD_PATH       = DATA_DIR / "nsd_func1pt8mm_individualROIs.h5"
NSD_LH_PATH    = DATA_DIR / "nsd-subj01-ncsnr-lh.mgh"
NSD_RH_PATH    = DATA_DIR / "nsd-subj01-ncsnr-rh.mgh"

# Model feature files
MODEL_A = "adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0"
MODEL_B = "Qwen3-VL-2B-Instruct"

FEAT_A_THINGS = FEAT_DIR / MODEL_A / "things_stimuli.h5"
FEAT_A_NSD    = FEAT_DIR / MODEL_A / "nsd_stimuli.h5"
FEAT_B_THINGS = FEAT_DIR / MODEL_B / "things_stimuli.h5"
FEAT_B_NSD    = FEAT_DIR / MODEL_B / "nsd_stimuli.h5"



# Usefull Functions

def print_h5_tree(path, max_depth=3, _prefix="", _depth=0):
    """Recursively print the keys and shapes of an HDF5 file."""
    with h5py.File(path, "r") as f:
        _recurse_h5(f, max_depth=max_depth)

def _recurse_h5(node, max_depth, _prefix="", _depth=0):
    for key in node.keys():
        item = node[key]
        if isinstance(item, h5py.Dataset):
            print(f"{_prefix}[Dataset] {key!r:30s}  shape={item.shape}  dtype={item.dtype}")
        elif isinstance(item, h5py.Group):
            print(f"{_prefix}[Group]   {key!r}")
            if _depth < max_depth:
                _recurse_h5(item, max_depth, _prefix + "  ", _depth + 1)


def load_h5_meta(path):
    """Return a dict of {key: shape} for every dataset in the file (top-level)."""
    meta = {}
    with h5py.File(path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                meta[name] = {"shape": obj.shape, "dtype": str(obj.dtype)}
        f.visititems(visitor)
    return meta


def build_id_index(ids_array):
    """
    Given a 1-D array of IDs (byte strings or ints),
    return a dict {id_value: row_index}.
    """
    return {id_: i for i, id_ in enumerate(ids_array)}


def align_ids(source_ids, target_index):
    """
    Map source_ids → row indices into a target array.
    Raises a KeyError if any ID is missing (fail-fast stimulus matching).
    """
    return np.array([target_index[x] for x in source_ids])


def verify_stimulus_match(neural_ids, feat_ids, label=""):
    """
    Assert that every neural ID has a matching feature ID.
    Prints a one-line summary on success; raises AssertionError on failure.
    """
    feat_set = set(feat_ids)
    missing  = [x for x in neural_ids if x not in feat_set]
    assert len(missing) == 0, (
        f"[{label}] {len(missing)} neural IDs have no matching feature row!"
    )
    print(f"  ✓ [{label}] All {len(neural_ids):,} neural IDs found in feature file.")




# Check Neural Datasets

print("=" * 60)
print("TVSD — Macaque Electrophysiology")
print("=" * 60)
print_h5_tree(TVSD_PATH)

In [6]:
print(TVSD_PATH,"\n",
EEG_PATH      ,"\n",
EEG_REPS_PATH  ,"\n",
NSD_PATH      , "\n",
NSD_LH_PATH   ,"\n",
NSD_RH_PATH  )

/shared/NX-414/data/tvsd.h5 
 /shared/NX-414/data/things_eeg2.h5 
 /shared/NX-414/data/things_eeg2-test_reps.h5 
 /shared/NX-414/data/nsd_func1pt8mm_individualROIs.h5 
 /shared/NX-414/data/nsd-subj01-ncsnr-lh.mgh 
 /shared/NX-414/data/nsd-subj01-ncsnr-rh.mgh


In [ ]:
print("\n" + "=" * 60)
print("THINGS-EEG2 — Human EEG")
print("=" * 60)
print_h5_tree(EEG_PATH)

In [ ]:
print("\n" + "=" * 60)
print("NSD — Human fMRI")
print("=" * 60)
print_h5_tree(NSD_PATH)

---

In [ ]:
# Load Dataset Metadata into Dicts for easy access

def inspect_neural_dataset(path):
    """
    Open an HDF5 neural dataset and return a summary dict.
    Handles TVSD, EEG, and NSD conventions.
    """
    info = {}
    with h5py.File(path, "r") as f:
        keys = list(f.keys())
        info["keys"] = keys

        # --- stimulus counts ---
        for split in ("train", "test"):
            id_key = f"{split}_ids"
            if id_key in f:
                info[f"n_{split}"] = f[id_key].shape[0]

        # --- neural response shape ---
        for split in ("train", "test"):
            x_key = f"{split}_X"
            if x_key in f:
                info[f"{split}_shape"] = f[x_key].shape

        # --- subjects (NSD / EEG) ---
        if "subjects" in f:
            info["subjects"] = list(f["subjects"].keys())
        elif "subject_ids" in f:
            info["subjects"] = [s.decode() if isinstance(s, bytes) else s
                                 for s in f["subject_ids"][:]]

        # --- ROIs (NSD) ---
        if "rois" in f:
            info["rois"] = list(f["rois"].keys())

        # --- noise ceiling ---
        for nc_key in ("noise_ceiling", "nc", "ncsnr"):
            if nc_key in f:
                info["noise_ceiling_shape"] = f[nc_key].shape
                break

    return info


tvsd_info = inspect_neural_dataset(TVSD_PATH)
eeg_info  = inspect_neural_dataset(EEG_PATH)
nsd_info  = inspect_neural_dataset(NSD_PATH)

print("TVSD info :", tvsd_info)
print("EEG  info :", eeg_info)
print("NSD  info :", nsd_info)

In [ ]:
# Inspect Feature Files

def inspect_feature_file(path):
    """Return layer names, feature shapes, and ID array from a feature HDF5."""
    info = {"layers": {}}
    with h5py.File(path, "r") as f:
        info["keys"] = list(f.keys())
        if "ids" in f:
            ids = f["ids"][:]
            info["n_stimuli"] = len(ids)
            info["id_dtype"]  = str(ids.dtype)
        for key in f.keys():
            if key == "ids":
                continue
            obj = f[key]
            if isinstance(obj, h5py.Dataset):
                info["layers"][key] = obj.shape
            elif isinstance(obj, h5py.Group):
                # layers stored as sub-groups
                for sub in obj.keys():
                    info["layers"][f"{key}/{sub}"] = obj[sub].shape
    return info


print("=" * 60)
print("Model A (ResNet-152 adversarial) — THINGS stimuli")
print("=" * 60)
feat_a_things_info = inspect_feature_file(FEAT_A_THINGS)
print(feat_a_things_info)

print("\nModel A (ResNet-152 adversarial) — NSD stimuli")
print("=" * 60)
feat_a_nsd_info = inspect_feature_file(FEAT_A_NSD)
print(feat_a_nsd_info)

print("\nModel B (Qwen3-VL) — THINGS stimuli")
print("=" * 60)
feat_b_things_info = inspect_feature_file(FEAT_B_THINGS)
print(feat_b_things_info)

print("\nModel B (Qwen3-VL) — NSD stimuli")
print("=" * 60)
feat_b_nsd_info = inspect_feature_file(FEAT_B_NSD)
print(feat_b_nsd_info)

In [ ]:
# Stimulus Matching Verification

print("\n--- Stimulus matching: THINGS-based datasets ---\n")

# Load IDs using the actual nested structure
with h5py.File(TVSD_PATH, "r") as f:
    tvsd_train_ids = f["train/stimulus_ids"][:]
    tvsd_test_ids  = f["test/stimulus_ids"][:]

with h5py.File(EEG_PATH, "r") as f:
    eeg_train_ids = f["train/stimulus_ids"][:]
    eeg_test_ids  = f["test/stimulus_ids"][:]

# Feature IDs (object dtype → decode if bytes for safe comparison)
with h5py.File(FEAT_A_THINGS, "r") as f:
    feat_things_ids_A = f["ids"][:]

with h5py.File(FEAT_B_THINGS, "r") as f:
    feat_things_ids_B = f["ids"][:]

# Verify all THINGS neural IDs are in feature files
verify_stimulus_match(tvsd_train_ids, feat_things_ids_A, "TVSD-train vs ModelA")
verify_stimulus_match(tvsd_test_ids,  feat_things_ids_A, "TVSD-test  vs ModelA")
verify_stimulus_match(eeg_train_ids,  feat_things_ids_A, "EEG-train  vs ModelA")
verify_stimulus_match(eeg_test_ids,   feat_things_ids_A, "EEG-test   vs ModelA")
verify_stimulus_match(tvsd_train_ids, feat_things_ids_B, "TVSD-train vs ModelB")
verify_stimulus_match(tvsd_test_ids,  feat_things_ids_B, "TVSD-test  vs ModelB")
verify_stimulus_match(eeg_train_ids,  feat_things_ids_B, "EEG-train  vs ModelB")
verify_stimulus_match(eeg_test_ids,   feat_things_ids_B, "EEG-test   vs ModelB")


print("\n--- Stimulus matching: NSD (per-subject integer IDs) ---\n")

with h5py.File(FEAT_A_NSD, "r") as f:
    feat_nsd_ids_A = f["ids"][:]          # shape (73000,), dtype int64

with h5py.File(FEAT_B_NSD, "r") as f:
    feat_nsd_ids_B = f["ids"][:]

feat_nsd_index_A = build_id_index(feat_nsd_ids_A)
feat_nsd_index_B = build_id_index(feat_nsd_ids_B)

# NSD stimulus IDs live at train/stimulus_ids/<subj> and test/stimulus_ids/<subj>
with h5py.File(NSD_PATH, "r") as f:
    nsd_subjects = list(f["train/stimulus_ids"].keys())   # ['subj01', ..., 'subj08']
    for subj in nsd_subjects:
        for split in ("train", "test"):
            subj_ids = f[f"{split}/stimulus_ids/{subj}"][:]
            verify_stimulus_match(subj_ids, feat_nsd_index_A, f"NSD {subj} {split} vs ModelA")
            verify_stimulus_match(subj_ids, feat_nsd_index_B, f"NSD {subj} {split} vs ModelB")

In [ ]:
# Pre-compute alignment indices for later sections

feat_things_index_A = build_id_index(feat_things_ids_A)
feat_things_index_B = build_id_index(feat_things_ids_B)

tvsd_train_feat_idx_A = align_ids(tvsd_train_ids, feat_things_index_A)
tvsd_test_feat_idx_A  = align_ids(tvsd_test_ids,  feat_things_index_A)
eeg_train_feat_idx_A  = align_ids(eeg_train_ids,  feat_things_index_A)
eeg_test_feat_idx_A   = align_ids(eeg_test_ids,   feat_things_index_A)

tvsd_train_feat_idx_B = align_ids(tvsd_train_ids, feat_things_index_B)
tvsd_test_feat_idx_B  = align_ids(tvsd_test_ids,  feat_things_index_B)
eeg_train_feat_idx_B  = align_ids(eeg_train_ids,  feat_things_index_B)
eeg_test_feat_idx_B   = align_ids(eeg_test_ids,   feat_things_index_B)

# Store NSD per-subject indices in dicts for easy lookup in later sections
nsd_train_feat_idx_A, nsd_test_feat_idx_A = {}, {}
nsd_train_feat_idx_B, nsd_test_feat_idx_B = {}, {}

with h5py.File(NSD_PATH, "r") as f:
    for subj in nsd_subjects:
        tr_ids = f[f"train/stimulus_ids/{subj}"][:]
        te_ids = f[f"test/stimulus_ids/{subj}"][:]
        nsd_train_feat_idx_A[subj] = align_ids(tr_ids, feat_nsd_index_A)
        nsd_test_feat_idx_A[subj]  = align_ids(te_ids, feat_nsd_index_A)
        nsd_train_feat_idx_B[subj] = align_ids(tr_ids, feat_nsd_index_B)
        nsd_test_feat_idx_B[subj]  = align_ids(te_ids, feat_nsd_index_B)

print("\nAll alignment indices computed and stored.")
print(f"  TVSD  train/test : {len(tvsd_train_feat_idx_A):,} / {len(tvsd_test_feat_idx_A):,}")
print(f"  EEG   train/test : {len(eeg_train_feat_idx_A):,} / {len(eeg_test_feat_idx_A):,}")
for subj in nsd_subjects:
    print(f"  NSD   {subj} train/test : "
          f"{len(nsd_train_feat_idx_A[subj]):,} / {len(nsd_test_feat_idx_A[subj]):,}")


In [ ]:
# Overview pf layers  

# Layer names from the actual feature files
layers_A = list(feat_a_things_info["layers"].keys())   # 10 layers
layers_B = list(feat_b_things_info["layers"].keys())   # 10 layers


# Also print the exact layer names for reference
print("\n--- Model A layers (10 total) ---")
for i, l in enumerate(layers_A): print(f"  [{i}] {l}")
print("\n--- Model B layers (10 total) ---")
for i, l in enumerate(layers_B): print(f"  [{i}] {l}")


<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 0</strong><br>Summarize the structure of the three datasets and the two feature sets in 4–6 sentences. Clearly state which dataset is most complex structurally and why.</div>

TVSD is the simplest dataset: a 2-D (stimuli × units) matrix from 2 macaques across 3 ventral-stream areas (V1, V4, IT), with 22,248 train and 100 test images shared across animals. THINGS-EEG2 adds a temporal axis, the 3-D tensor (stimuli × channels × 80 time points) is organized by 7 scalp-region groups across 10 subjects, and a separate file preserves individual repetitions for noise-ceiling estimation. NSD is structurally the most complex: 8 subjects each saw a different, partially overlapping subset of 8,302–9,000 train and 907–1,000 test images (drawn from a 73,000-image pool), with responses organized across 28 distinct visual ROIs of varying size, requiring per-subject stimulus-index look-ups before any analysis. Both feature sets (ResNet-152 adversarial and Qwen3-VL-2B) provide 10 layers each, all projected to 30,000 dimensions; the THINGS feature files cover 26,107 stimuli while the NSD files cover all 73,000, so per-subject index alignment is essential for NSD. NSD is most complex because it combines the largest stimulus pool, subject-specific non-overlapping subsets, the most ROIs (28), and the most diverse voxel counts across subjects.

### Datasets

| Dataset       | Modality                  | N  | Train        | Test       | ROIs              |
|--------------|--------------------------|----|-------------|-----------|------------------|
| TVSD         | Macaque electrophysiology | 2  | 22,248       | 100        | V1, V4, IT       |
| THINGS-EEG2  | Human EEG                | 10 | 16,540       | 200        | 7 region groups  |
| NSD          | Human fMRI               | 8  | 8,302–9,000  | 907–1,000  | 28 visual ROIs   |

### Response structure

| Dataset       | Shape                              | Channels                | Time                     | Repetitions        |
|--------------|------------------------------------|-------------------------|--------------------------|--------------------|
| TVSD         | (100, n_units) per animal×ROI      | —                       | —                        | —                  |
| THINGS-EEG2  | (200, n_ch, 80) per subject×region | 3–63 per region         | 80 @ 100 Hz (0–0.8 s)    | Yes (test_reps)    |
| NSD          | (n_stim, n_voxels) per subject×ROI | —                       | Single beta              | Averaged           |

### Models

| Model                  | Type                          | Train data                     | Output shape                  |
|-----------------------|-------------------------------|--------------------------------|-------------------------------|
| ResNet-152            | CNN (ImageNet, adversarial)   | 26,107 / 73,000                | (n_stim, 30k) × 10 layers     |
| Qwen3-VL-2B           | Vision-language transformer   | 26,107 / 73,000                | (n_stim, 30k) × 10 layers     |

# 1. Inspection, Visualization, and Noise Ceiling Estimates

This section is about understanding the data before doing model comparison. By the end of it, you should have a clear sense of how each modality is organized, which signals appear reliable, and how the provided reliability estimates relate to your own computations.

## 1.1 Inspect the datasets

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Inspect the content and axis meaning of TVSD, EEG2, and NSD.
- Identify where subject IDs, ROI labels, time axes, stimulus IDs, and noise ceilings are stored.
- State clearly what each axis means for each response array.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **TVSD structure table**
2. **EEG2 structure table**
3. **NSD structure table**

Each table must list the array name, array shape, and the meaning of each axis.

In [10]:
from IPython.display import display, HTML
import pandas as pd
import h5py

# ── Shared CSS injected once ──────────────────────────────────────────────────
display(HTML("""
<style>
  .neuro-section {
    font-family: 'IBM Plex Mono', 'Fira Code', 'Courier New', monospace;
    margin: 2em 0 2.5em 0;
  }
  .neuro-title {
    font-family: 'IBM Plex Sans', 'Segoe UI', sans-serif;
    font-size: 0.75rem;
    font-weight: 700;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    color: #94a3b8;
    margin-bottom: 0.4em;
  }
  .neuro-heading {
    font-family: 'IBM Plex Sans', 'Segoe UI', sans-serif;
    font-size: 1.1rem;
    font-weight: 600;
    color: #1e293b;
    margin: 0 0 0.8em 0;
    padding-bottom: 0.4em;
    border-bottom: 2px solid #e2e8f0;
    display: flex;
    align-items: center;
    gap: 0.5em;
  }
  .neuro-heading .badge {
    font-size: 0.65rem;
    font-weight: 700;
    padding: 0.2em 0.55em;
    border-radius: 999px;
    letter-spacing: 0.08em;
    text-transform: uppercase;
  }
  .badge-ephys  { background: #fef3c7; color: #92400e; }
  .badge-eeg    { background: #dbeafe; color: #1e40af; }
  .badge-fmri   { background: #dcfce7; color: #166534; }

  .neuro-table {
    width: max-content;
    min-width: 100%;
    border-collapse: collapse;
    font-size: 0.8rem;
    background: white;
    border-radius: 8px;
    overflow: hidden;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08), 0 0 0 1px rgba(0,0,0,0.06);
  }
  .neuro-section {
    overflow-x: auto;
  }
  .neuro-table thead tr {
    background: #f8fafc;
  }
  .neuro-table thead th {
    font-family: 'IBM Plex Sans', sans-serif;
    font-size: 0.7rem;
    font-weight: 600;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    color: #64748b;
    padding: 0.65em 0.9em;
    text-align: left;
    border-bottom: 1px solid #e2e8f0;
    white-space: nowrap;
  }
  .neuro-table tbody tr {
    border-bottom: 1px solid #f1f5f9;
    transition: background 0.12s;
  }
  .neuro-table tbody tr:last-child { border-bottom: none; }
  .neuro-table tbody tr:hover { background: #f8fafc; }
  .neuro-table td {
    padding: 0.55em 0.9em;
    vertical-align: top;
    color: #334155;
  }

  /* Column-specific styles */
  .col-path  { color: #6d28d9; font-weight: 500; white-space: nowrap; }
  .col-shape {
    font-weight: 700;
    color: #0f172a;
    background: #f8fafc;
    white-space: nowrap;
  }
  .col-notes { color: #64748b; font-style: italic; font-size: 0.75rem; }
  .col-axis  { color: #475569; font-size: 0.77rem; }

  /* Row striping by group (monkey / subject) */
  .neuro-table tbody tr.group-alt { background: #fafafa; }
</style>
"""))


# ── Helper: render one table ──────────────────────────────────────────────────
def _row_html(row, col_classes):
    cells = "".join(
        f'<td class="{col_classes.get(c, "")}">{v if pd.notna(v) and v != "—" else "<span style=\"color:#cbd5e1\">—</span>"}</td>'
        for c, v in row.items()
    )
    return f"<tr>{cells}</tr>"


def render_table(title, subtitle, badge_label, badge_class, df, col_classes=None):
    col_classes = col_classes or {}
    headers = "".join(f"<th>{c}</th>" for c in df.columns)
    rows    = "".join(_row_html(row, col_classes) for _, row in df.iterrows())

    html = f"""
    <div class="neuro-section">
      <div class="neuro-title">{subtitle}</div>
      <div class="neuro-heading">
        {title}
        <span class="badge {badge_class}">{badge_label}</span>
      </div>
      <table class="neuro-table">
        <thead><tr>{headers}</tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
    """
    display(HTML(html))


# ── Google Font import (optional, silently ignored if offline) ────────────────
display(HTML('<link rel="preconnect" href="https://fonts.googleapis.com">'
             '<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;700&family=IBM+Plex+Sans:wght@400;600;700&display=swap" rel="stylesheet">'))


# ════════════════════════════════════════════════════════════════════════════════
# TABLE 1 — TVSD  Macaque Electrophysiology
# ════════════════════════════════════════════════════════════════════════════════
tvsd_rows = []
with h5py.File(TVSD_PATH, "r") as f:
    for monkey in ["monkeyF", "monkeyN"]:
        for roi in ["V1", "V4", "IT"]:
            for split in ["train", "test"]:
                arr = f[f"{split}/neural_data/{monkey}/{roi}"]
                tvsd_rows.append({
                    "Path": f"{split}/neural_data/{monkey}/{roi}",
                    "Shape": str(arr.shape),
                    "Axis 0": f"stimuli ({arr.shape[0]})",
                    "Axis 1": f"units ({arr.shape[1]})",
                    "Monkey": monkey,
                    "ROI": roi,
                    "Split": split,
                })
    for split in ["train", "test"]:
        arr = f[f"{split}/stimulus_ids"]
        tvsd_rows.append({
            "Path": f"{split}/stimulus_ids",
            "Shape": str(arr.shape),
            "Axis 0": "stimuli (byte-string IDs)",
            "Axis 1": "—",
            "Monkey": "—", "ROI": "—", "Split": split,
        })
    for monkey in ["monkeyF", "monkeyN"]:
        for roi in ["V1", "V4", "IT"]:
            arr = f[f"noise_ceilings/{monkey}/{roi}"]
            tvsd_rows.append({
                "Path": f"noise_ceilings/{monkey}/{roi}",
                "Shape": str(arr.shape),
                "Axis 0": f"units ({arr.shape[0]}) · % reliability ÷100",
                "Axis 1": "—",
                "Monkey": monkey, "ROI": roi, "Split": "—",
            })

tvsd_df = pd.DataFrame(tvsd_rows)
render_table(
    title="TVSD — Macaque Electrophysiology",
    subtitle="Table 1",
    badge_label="Ephys",
    badge_class="badge-ephys",
    df=tvsd_df,
    col_classes={"Path": "col-path", "Shape": "col-shape",
                 "Axis 0": "col-axis", "Axis 1": "col-axis"}
)


# ════════════════════════════════════════════════════════════════════════════════
# TABLE 2 — THINGS-EEG2  Human EEG
# ════════════════════════════════════════════════════════════════════════════════
eeg_rows = []
with h5py.File(EEG_PATH, "r") as f:
    subj = "sub-01"
    roi  = "whole_brain"
    for split in ["train", "test"]:
        arr = f[f"{split}/neural_data/{subj}/{roi}"]
        eeg_rows.append({
            "Path": f"{split}/neural_data/{subj}/{roi}",
            "Shape": str(arr.shape),
            "Axis 0": f"stimuli ({arr.shape[0]})",
            "Axis 1": f"channels ({arr.shape[1]})",
            "Axis 2": f"time pts ({arr.shape[2]}, 0–0.8 s @ 100 Hz)",
            "Subject": subj, "ROI": roi, "Split": split,
        })
    arr = f["test/stimulus_ids"]
    eeg_rows.append({
        "Path": "test/stimulus_ids",
        "Shape": str(arr.shape),
        "Axis 0": "stimuli (byte-string IDs)",
        "Axis 1": "—", "Axis 2": "—",
        "Subject": "all", "ROI": "—", "Split": "test",
    })
    arr = f[f"noise_ceilings/{subj}/{roi}"]
    eeg_rows.append({
        "Path": f"noise_ceilings/{subj}/{roi}",
        "Shape": str(arr.shape),
        "Axis 0": f"channels ({arr.shape[0]})",
        "Axis 1": f"time pts ({arr.shape[1]}) · % reliability ÷100",
        "Axis 2": "—",
        "Subject": subj, "ROI": roi, "Split": "—",
    })

eeg_df = pd.DataFrame(eeg_rows)
render_table(
    title="THINGS-EEG2 — Human EEG",
    subtitle="Table 2",
    badge_label="EEG",
    badge_class="badge-eeg",
    df=eeg_df,
    col_classes={"Path": "col-path", "Shape": "col-shape",
                 "Axis 0": "col-axis", "Axis 1": "col-axis", "Axis 2": "col-axis"}
)


# ════════════════════════════════════════════════════════════════════════════════
# TABLE 3 — NSD  Human fMRI
# ════════════════════════════════════════════════════════════════════════════════
nsd_rows = []
with h5py.File(NSD_PATH, "r") as f:
    subj = "subj01"
    roi  = "nsdgeneral"
    for split in ["train", "test"]:
        arr = f[f"{split}/neural_data/{subj}/{roi}"]
        nsd_rows.append({
            "Path": f"{split}/neural_data/{subj}/{roi}",
            "Shape": str(arr.shape),
            "Axis 0": f"stimuli ({arr.shape[0]})",
            "Axis 1": f"voxels ({arr.shape[1]})",
            "Subject": subj, "ROI": roi, "Split": split,
        })
    arr = f[f"test/stimulus_ids/{subj}"]
    nsd_rows.append({
        "Path": f"test/stimulus_ids/{subj}",
        "Shape": str(arr.shape),
        "Axis 0": "stimuli (integer NSD IDs)",
        "Axis 1": "subject-specific subset of 73 k pool",
        "Subject": subj, "ROI": "—", "Split": "test",
    })
    arr = f[f"noise_ceilings/{subj}/{roi}"]
    nsd_rows.append({
        "Path": f"noise_ceilings/{subj}/{roi}",
        "Shape": str(arr.shape),
        "Axis 0": f"voxels ({arr.shape[0]}) · % reliability ÷100",
        "Axis 1": "—",
        "Subject": subj, "ROI": roi, "Split": "—",
    })

nsd_df = pd.DataFrame(nsd_rows)
render_table(
    title="NSD — Human fMRI (Example for subject 01)",
    subtitle="Table 3",
    badge_label="fMRI",
    badge_class="badge-fmri",
    df=nsd_df,
    col_classes={"Path": "col-path", "Shape": "col-shape",
                 "Axis 0": "col-axis", "Axis 1": "col-axis"}
)

Path,Shape,Axis 0,Axis 1,Monkey,ROI,Split
train/neural_data/monkeyF/V1,"(22248, 462)",stimuli (22248),units (462),monkeyF,V1,train
test/neural_data/monkeyF/V1,"(100, 462)",stimuli (100),units (462),monkeyF,V1,test
train/neural_data/monkeyF/V4,"(22248, 139)",stimuli (22248),units (139),monkeyF,V4,train
test/neural_data/monkeyF/V4,"(100, 139)",stimuli (100),units (139),monkeyF,V4,test
train/neural_data/monkeyF/IT,"(22248, 241)",stimuli (22248),units (241),monkeyF,IT,train
test/neural_data/monkeyF/IT,"(100, 241)",stimuli (100),units (241),monkeyF,IT,test
train/neural_data/monkeyN/V1,"(22248, 437)",stimuli (22248),units (437),monkeyN,V1,train
test/neural_data/monkeyN/V1,"(100, 437)",stimuli (100),units (437),monkeyN,V1,test
train/neural_data/monkeyN/V4,"(22248, 247)",stimuli (22248),units (247),monkeyN,V4,train
test/neural_data/monkeyN/V4,"(100, 247)",stimuli (100),units (247),monkeyN,V4,test


Path,Shape,Axis 0,Axis 1,Axis 2,Subject,ROI,Split
train/neural_data/sub-01/whole_brain,"(16540, 63, 80)",stimuli (16540),channels (63),"time pts (80, 0–0.8 s @ 100 Hz)",sub-01,whole_brain,train
test/neural_data/sub-01/whole_brain,"(200, 63, 80)",stimuli (200),channels (63),"time pts (80, 0–0.8 s @ 100 Hz)",sub-01,whole_brain,test
test/stimulus_ids,"(200,)",stimuli (byte-string IDs),—,—,all,—,test
noise_ceilings/sub-01/whole_brain,"(63, 80)",channels (63),time pts (80) · % reliability ÷100,—,sub-01,whole_brain,—


Path,Shape,Axis 0,Axis 1,Subject,ROI,Split
train/neural_data/subj01/nsdgeneral,"(9000, 12687)",stimuli (9000),voxels (12687),subj01,nsdgeneral,train
test/neural_data/subj01/nsdgeneral,"(1000, 12687)",stimuli (1000),voxels (12687),subj01,nsdgeneral,test
test/stimulus_ids/subj01,"(1000,)",stimuli (integer NSD IDs),subject-specific subset of 73 k pool,subj01,—,test
noise_ceilings/subj01/nsdgeneral,"(12687,)",voxels (12687) · % reliability ÷100,—,subj01,nsdgeneral,—


<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.1</strong><br>Explain the main differences between the three modalities in terms of what is being measured and how the data are organized.</div>

TVSD records extracellular multi-unit firing rates (a proxy for spiking activity) from implanted electrodes in two macaques, producing a flat 2-D matrix of (stimuli × units) per area; it has the highest spatial specificity at the single-neuron level but no temporal resolution beyond the analysis window. THINGS-EEG2 records scalp electric potentials with millisecond resolution, yielding a 3-D tensor (stimuli × channels × 80 time points) that captures the temporal dynamics of the visual evoked response but cannot resolve individual neurons or cortical depth. NSD records BOLD signal (an indirect, haemodynamic proxy for neural activity) with millimetre spatial resolution across 28 labeled fMRI ROIs, providing the richest anatomical coverage but no temporal information — each stimulus produces a single beta estimate per voxel. The three datasets therefore span a spatial-temporal trade-off: electrophysiology is unit-precise but temporally averaged, EEG has high temporal resolution but poor spatial resolution, and fMRI has high spatial resolution but collapses time into a single response.

---

## 1.2 Visualize EEG signals

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Plot example EEG responses for several stimuli and channels.
- Plot average responses over time for at least one subject and one ROI.
- Visualize the provided EEG noise ceilings over channels and time.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One plot of example EEG time courses** for several stimuli and channels.
2. **One heatmap over channels × time** for at least one subject and one ROI.
3. **One summary plot of the provided EEG noise ceilings**.
4. **One short written interpretation** in Answer box 1.2.

In [ ]:
# ============================================================
# 1.2  EEG Visualization
# ============================================================

SUBJ    = "sub-01"
ROI     = "occipital"       # small (3 channels) → cleaner for illustration
ROI_BIG = "whole_brain"     # 63 channels → good for heatmap
TIME    = np.linspace(0, 0.8, 80)  # 80 points @ 100 Hz

# Load test data for visualization (200 stimuli)
with h5py.File(EEG_PATH, "r") as f:
    eeg_test_occ  = f[f"test/neural_data/{SUBJ}/{ROI}"][:]        # (200, 3, 80)
    eeg_test_wb   = f[f"test/neural_data/{SUBJ}/{ROI_BIG}"][:]    # (200, 63, 80)
    nc_wb         = f[f"noise_ceilings/{SUBJ}/{ROI_BIG}"][:] / 100.0  # (63, 80)

# ---------- Plot 1: Example time courses for 5 stimuli × 3 channels ----------
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
n_stim_show = 8
colors = plt.cm.tab10(np.linspace(0, 1, n_stim_show))

for ch_idx, ax in enumerate(axes):
    for si in range(n_stim_show):
        ax.plot(TIME, eeg_test_occ[si, ch_idx, :],
                color=colors[si], alpha=0.75, lw=1.2,
                label=f"stim {si}" if ch_idx == 0 else None)
    # plot mean over stimuli
    ax.plot(TIME, eeg_test_occ[:, ch_idx, :].mean(axis=0),
            color="black", lw=2.2, label="mean" if ch_idx == 0 else None)
    ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.set_ylabel(f"Ch {ch_idx+1}\n(μV)")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].legend(ncol=5, fontsize=7, loc="upper right")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"EEG example traces — {SUBJ}, ROI: {ROI}", fontsize=12)
plt.tight_layout()
plt.savefig("eeg_example_traces.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- Plot 2: Channel × Time heatmap (mean over stimuli) ----------
mean_response = eeg_test_wb.mean(axis=0)   # (63, 80)

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(mean_response, aspect="auto",
               extent=[TIME[0], TIME[-1], mean_response.shape[0], 0],
               cmap="RdBu_r", vmin=-np.percentile(np.abs(mean_response), 98),
               vmax= np.percentile(np.abs(mean_response), 98))
plt.colorbar(im, ax=ax, label="Amplitude (μV)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Channel index (whole_brain)")
ax.set_title(f"Mean EEG response — {SUBJ}, whole_brain (n=63 channels, n=200 stimuli)")
ax.axvline(0.1, color="white", lw=0.8, ls="--", label="100 ms")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("eeg_channel_time_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- Plot 3: Provided noise ceilings (whole_brain, sub-01) ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap: channels × time
im = axes[0].imshow(nc_wb, aspect="auto",
                    extent=[TIME[0], TIME[-1], nc_wb.shape[0], 0],
                    cmap="viridis", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[0], label="Noise ceiling [0,1]")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Channel index")
axes[0].set_title(f"Provided NC — {SUBJ}, whole_brain")

# Mean over channels → time profile
axes[1].plot(TIME, nc_wb.mean(axis=0), color="steelblue", lw=2, label="mean over channels")
axes[1].fill_between(TIME,
                     nc_wb.mean(axis=0) - nc_wb.std(axis=0),
                     nc_wb.mean(axis=0) + nc_wb.std(axis=0),
                     alpha=0.25, color="steelblue")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Noise ceiling [0,1]")
axes[1].set_title("Mean ± std noise ceiling over time")
axes[1].spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("eeg_nc_provided.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 1.3 Estimate EEG noise ceilings using two methods

In practice, there are multiple ways to estimate noise ceilings, depending on the available data and the specific research question. When you have repeated measurements of the same stimulus, you can estimate reliability from the consistency of those repetitions. When repeated measurements are not available, reliability can instead be estimated across subjects, which often yields a more conservative ceiling.

In this part, you will implement two different estimators using the `things_eeg2-test_reps.h5` file, which contains the unaveraged test responses with repetitions.

You must implement **two different estimators** using `things_eeg2-test_reps.h5`. You can refer to the cited paper in each method's docstring for details.


### Required estimators

1. **Variance-based estimator**  
2. **Split-half reliability estimator**

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Implement both estimators.
- Compute both estimators for EEG2.
- Compare them to the provided EEG noise ceilings stored in `things_eeg2.h5`.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Working implementation of the variance-based estimator**
2. **Working implementation of the split-half estimator**
3. **One plot of mean noise ceiling over time**
4. **One plot of mean noise ceiling over channels**
5. **At least one channel × time heatmap for each estimator**
6. **At least one histogram comparing the value distributions**
7. **One direct visual comparison to the stored EEG noise ceilings**

### Starter functions

In [ ]:
# ============================================================
# 1.3  Noise Ceiling Estimators
# ============================================================

def compute_ceiling_variancebased(
    responses: np.ndarray,
    nan_policy: str = "omit"
) -> np.ndarray:
    """
    Variance-based noise ceiling (Allen et al. 2022 / NSD-style).

    Input shape: (n_channels, n_timepoints, n_stimuli, n_reps)
    or            (n_units, n_stimuli, n_reps)

    Steps:
      1) z-score across stimuli for each (unit, rep)  → total var ≈ 1
      2) noise_var = mean over stimuli of var over reps
      3) signal_var = 1 - noise_var
      4) snr = signal_var / noise_var   (clipped to ≥ 0)
      5) nc  = 100 * snr / (snr + 1/n_reps)
    """
    # responses: (..., n_stimuli, n_reps)
    n_reps = responses.shape[-1]
    n_stim = responses.shape[-2]
    leading = responses.shape[:-2]   # (n_ch, n_tp) or (n_units,)

    # 1) z-score over stimuli (axis=-2) for each unit × rep
    mu  = responses.mean(axis=-2, keepdims=True)
    std = responses.std( axis=-2, keepdims=True)
    std = np.where(std == 0, 1.0, std)         # avoid div-by-zero
    z   = (responses - mu) / std               # same shape

    # 2) noise variance: var over reps, then mean over stimuli
    noise_var = z.var(axis=-1, ddof=1).mean(axis=-1)   # shape: leading

    # 3) signal variance (total var ≈ 1 after z-scoring)
    signal_var = np.clip(1.0 - noise_var, 0, None)

    # 4) SNR
    snr = np.where(noise_var > 0, signal_var / noise_var, 0.0)

    # 5) reliability as percent
    nc = 100.0 * snr / (snr + 1.0 / n_reps)
    return nc   # shape: leading = (n_ch, n_tp) or (n_units,)


def compute_ceiling_splithalf(
    responses: np.ndarray,
    folds: int = 10,
    seed: int = 0,
    spearman_brown: bool = True,
    equalize_halves: bool = True,
    clip_folds: bool = False
) -> np.ndarray:
    """
    Split-half reliability per unit (van Bree et al. 2025).

    Input shape: (..., n_stimuli, n_reps)
    Returns:     shape (...)   — values in [0, 100]

    Steps per fold:
      1) shuffle repetition indices, split into two halves
      2) average each half across reps → two response vectors per unit
      3) Pearson r across stimuli for each unit
      4) Spearman-Brown correction:  r_sb = 2r / (1 + r)
    Average across folds, return as percent.
    """
    n_reps = responses.shape[-1]
    n_stim = responses.shape[-2]
    leading_shape = responses.shape[:-2]

    # flatten leading dims for vectorised correlation
    flat = responses.reshape(-1, n_stim, n_reps)   # (n_units, n_stim, n_reps)
    n_units = flat.shape[0]

    half = n_reps // 2 if equalize_halves else (n_reps + 1) // 2
    rng  = np.random.default_rng(seed)

    fold_rs = np.zeros((folds, n_units), dtype=np.float64)

    for fi in range(folds):
        perm  = rng.permutation(n_reps)
        idx_a = perm[:half]
        idx_b = perm[half: 2*half] if equalize_halves else perm[half:]

        ha = flat[:, :, idx_a].mean(axis=-1)   # (n_units, n_stim)
        hb = flat[:, :, idx_b].mean(axis=-1)

        # Pearson r per unit (vectorised)
        ha_c = ha - ha.mean(axis=1, keepdims=True)
        hb_c = hb - hb.mean(axis=1, keepdims=True)
        num  = (ha_c * hb_c).sum(axis=1)
        den  = np.sqrt((ha_c**2).sum(axis=1) * (hb_c**2).sum(axis=1))
        r    = np.where(den > 0, num / den, 0.0)

        if spearman_brown:
            r = 2.0 * r / (1.0 + np.abs(r))   # abs avoids issues near -1

        if clip_folds:
            r = np.clip(r, 0, 1)

        fold_rs[fi] = r

    nc = fold_rs.mean(axis=0) * 100.0   # percent
    return nc.reshape(leading_shape)


# ============================================================
# Load repeated EEG test responses
# ============================================================
# Structure: test_reps file has shape (n_subjects, n_channels, n_timepoints, n_stimuli, n_reps)
# or a grouped h5 — let's inspect first

print("EEG test reps file structure:")
with h5py.File(EEG_REPS_PATH, "r") as f:
    def _show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name:60s}  {obj.shape}  {obj.dtype}")
    f.visititems(_show)

In [ ]:
# ============================================================
# 1.3  Noise Ceiling Estimators
#      test_reps shape: (n_stimuli, n_channels, n_timepoints, n_reps)
#                        = (200, 63, 80, 80)
# ============================================================

SUBJ_NC = "sub-01"
ROI_NC  = "whole_brain"

with h5py.File(EEG_REPS_PATH, "r") as f:
    reps_raw = f[f"test/neural_data/{SUBJ_NC}/{ROI_NC}"][:]

print(f"Raw reps shape: {reps_raw.shape}")
# Expected: (200, 63, 80, 80) = (n_stim, n_ch, n_tp, n_reps)

n_stim, n_ch, n_tp, n_reps = reps_raw.shape

# Reshape to (n_ch, n_tp, n_stim, n_reps) for both estimators
reps = reps_raw.transpose(1, 2, 0, 3)   # (63, 80, 200, 80)
print(f"Transposed reps shape: {reps.shape}  "
      f"(n_ch={n_ch}, n_tp={n_tp}, n_stim={n_stim}, n_reps={n_reps})")

# ============================================================
# Compute both estimators
# ============================================================
print("\nComputing variance-based ceiling...")
nc_var = compute_ceiling_variancebased(reps)   # (63, 80)
print(f"  nc_var:  shape={nc_var.shape},  mean={nc_var.mean():.2f}%  "
      f"min={nc_var.min():.2f}%  max={nc_var.max():.2f}%")

print("Computing split-half ceiling...")
nc_sh  = compute_ceiling_splithalf(reps, folds=10, seed=42)   # (63, 80)
print(f"  nc_sh:   shape={nc_sh.shape},   mean={nc_sh.mean():.2f}%  "
      f"min={nc_sh.min():.2f}%  max={nc_sh.max():.2f}%")

# Load provided ceiling (already in %, keep consistent)
with h5py.File(EEG_PATH, "r") as f:
    nc_provided = f[f"noise_ceilings/{SUBJ_NC}/{ROI_NC}"][:]   # (63, 80), percent
print(f"  nc_provided: shape={nc_provided.shape}, mean={nc_provided.mean():.2f}%")

# ============================================================
# Visualize — 6-panel figure
# ============================================================
TIME = np.linspace(0, 0.8, 80)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

vmax_all = np.percentile(
    np.concatenate([nc_var.ravel(), nc_sh.ravel(), nc_provided.ravel()]), 99
)
kw_heat = dict(aspect="auto",
               extent=[TIME[0], TIME[-1], n_ch, 0],
               cmap="viridis", vmin=0, vmax=vmax_all)

# --- Row 0: channel × time heatmaps ---
for ax, data, title in zip(
    axes[0],
    [nc_var, nc_sh, nc_provided],
    ["Variance-based (computed)", "Split-half SB (computed)", "Provided (stored)"]
):
    im = ax.imshow(data, **kw_heat)
    plt.colorbar(im, ax=ax, label="NC (%)")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channel index")

# --- Row 1, col 0: mean over channels vs time ---
ax = axes[1][0]
ax.plot(TIME, nc_var.mean(axis=0),      color="steelblue", lw=2,   label="Variance-based")
ax.fill_between(TIME,
                nc_var.mean(axis=0) - nc_var.std(axis=0),
                nc_var.mean(axis=0) + nc_var.std(axis=0),
                alpha=0.2, color="steelblue")
ax.plot(TIME, nc_sh.mean(axis=0),       color="tomato",    lw=2,   label="Split-half")
ax.fill_between(TIME,
                nc_sh.mean(axis=0) - nc_sh.std(axis=0),
                nc_sh.mean(axis=0) + nc_sh.std(axis=0),
                alpha=0.2, color="tomato")
ax.plot(TIME, nc_provided.mean(axis=0), color="black",     lw=2.5,
        ls="--", label="Provided")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Mean NC (%)")
ax.set_title("Mean NC over time (± std across channels)")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

# --- Row 1, col 1: mean over time vs channel ---
ax = axes[1][1]
ax.plot(nc_var.mean(axis=1),      np.arange(n_ch),
        color="steelblue", lw=2, label="Variance-based")
ax.plot(nc_sh.mean(axis=1),       np.arange(n_ch),
        color="tomato",    lw=2, label="Split-half")
ax.plot(nc_provided.mean(axis=1), np.arange(n_ch),
        color="black",     lw=2.5, ls="--", label="Provided")
ax.invert_yaxis()
ax.set_ylabel("Channel index"); ax.set_xlabel("Mean NC (%)")
ax.set_title("Mean NC per channel (averaged over time)")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

# --- Row 1, col 2: histogram ---
ax = axes[1][2]
bins = np.linspace(0, vmax_all + 5, 60)
ax.hist(nc_var.ravel(),      bins=bins, alpha=0.55,
        color="steelblue", label=f"Variance-based (μ={nc_var.mean():.1f}%)")
ax.hist(nc_sh.ravel(),       bins=bins, alpha=0.55,
        color="tomato",    label=f"Split-half (μ={nc_sh.mean():.1f}%)")
ax.hist(nc_provided.ravel(), bins=bins, alpha=0.55,
        color="black",     label=f"Provided (μ={nc_provided.mean():.1f}%)")
ax.set_xlabel("NC (%)"); ax.set_ylabel("Count")
ax.set_title("NC value distribution")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

plt.suptitle(f"Noise ceiling comparison — {SUBJ_NC}, {ROI_NC}", fontsize=13)
plt.tight_layout()
plt.savefig("eeg_nc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.3</strong><br>Compare the two estimators. Do they produce similar patterns across channels and time? Where do they differ most?</div>

Both estimators recover the same qualitative spatial-temporal structure as the provided ceilings: reliability peaks between 100–400 ms and is concentrated in two distinct channel groups (~channels 12–18 and ~44–52), which correspond to posterior scalp electrodes capturing the visual evoked response. However, the two estimators differ substantially in their quantitative agreement with the stored values. The variance-based estimator tracks the provided ceilings almost perfectly across all channel × time bins, while the split-half estimator systematically underestimates, with a mean bias of −3.4% and considerable scatter especially near zero — visible as the downward spread of orange residuals in the right panel. The split-half underestimation is most severe at low-to-moderate reliability bins (stored NC 0–10%), where sampling variability in the 40-repetition halves is largest and the Spearman-Brown correction cannot fully compensate.

---

## 1.4 Compare the noise ceiling estimators statistically on EEG2

In `things_eeg2.h5`, we provided noise ceilings computed using one of the two methods you implemented. Can you determine which one it is by comparing the stored ceilings to your computed ones?

Perform a hypothesis test to compare the stored ceilings to each of your computed estimators. For example, you could compute the mean squared error between the stored ceilings and each estimator per subject/time/channel, and then use a paired t-test to see if one estimator is significantly closer to the stored values than the other.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- State clearly what each estimator assumes.
- Define a quantitative comparison to the stored EEG noise ceilings.
- Run at least one simple statistical test or formal comparison.

Examples:

- mean absolute deviation from the stored values,
- paired comparison across channel × time units,
- correlation with the stored values.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One quantitative comparison table** comparing both estimators to the stored EEG noise ceilings.
2. **One statistical test or one formal quantitative comparison** such as a paired test, correlation analysis, or mean absolute deviation analysis.
3. **One concise written conclusion** stating which estimator better matches the stored values and why.

In [ ]:
# ============================================================
# 1.4  Statistical comparison to stored noise ceilings
# ============================================================
from scipy.stats import pearsonr, ttest_rel

flat_var  = nc_var.ravel()
flat_sh   = nc_sh.ravel()
flat_prov = nc_provided.ravel()

def compare_to_stored(estimated, stored, name):
    mae  = np.abs(estimated - stored).mean()
    rmse = np.sqrt(((estimated - stored) ** 2).mean())
    r, p = pearsonr(estimated, stored)
    bias = (estimated - stored).mean()   # signed: positive = overestimate
    return {"Estimator": name,
            "MAE (%)": round(mae, 3),
            "RMSE (%)": round(rmse, 3),
            "Pearson r": round(r, 4),
            "p-value": f"{p:.2e}",
            "Bias (%)": round(bias, 3)}

cmp_df = pd.DataFrame([
    compare_to_stored(flat_var, flat_prov, "Variance-based"),
    compare_to_stored(flat_sh,  flat_prov, "Split-half (SB)"),
]).set_index("Estimator")

print("=" * 65)
print("Quantitative comparison to stored noise ceilings")
print("=" * 65)
print(cmp_df.to_string())

# Paired t-test on absolute errors
err_var = np.abs(flat_var - flat_prov)
err_sh  = np.abs(flat_sh  - flat_prov)
t_stat, p_val = ttest_rel(err_var, err_sh)
winner = "Variance-based" if err_var.mean() < err_sh.mean() else "Split-half"
print(f"\nPaired t-test |err_var| vs |err_sh|: t={t_stat:.3f}, p={p_val:.2e}")
print(f"Lower MAE → {winner} is closer to stored values")

# ---- Scatter + residual plots ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, est, name, color in zip(
    axes[:2],
    [flat_var, flat_sh],
    ["Variance-based", "Split-half (SB)"],
    ["steelblue", "tomato"]
):
    ax.scatter(flat_prov, est, alpha=0.12, s=4, color=color, rasterized=True)
    lim = [min(flat_prov.min(), est.min()) - 1,
           max(flat_prov.max(), est.max()) + 1]
    ax.plot(lim, lim, "k--", lw=1.2, label="y = x")
    r, _ = pearsonr(est, flat_prov)
    ax.set_xlabel("Stored NC (%)")
    ax.set_ylabel(f"{name} (%)")
    ax.set_title(f"{name}\nr={r:.4f}  MAE={np.abs(est-flat_prov).mean():.2f}%")
    ax.legend(fontsize=8)
    ax.spines[["top","right"]].set_visible(False)

# Residual comparison
axes[2].axhline(0, color="black", lw=1, ls="--")
axes[2].plot(flat_prov, flat_var - flat_prov,
             "o", color="steelblue", alpha=0.15, ms=2,
             label="Variance-based residual")
axes[2].plot(flat_prov, flat_sh  - flat_prov,
             "o", color="tomato",    alpha=0.15, ms=2,
             label="Split-half residual")
axes[2].set_xlabel("Stored NC (%)")
axes[2].set_ylabel("Estimator − Stored (%)")
axes[2].set_title("Residuals vs stored NC")
axes[2].legend(fontsize=8, markerscale=3)
axes[2].spines[["top","right"]].set_visible(False)

plt.suptitle(f"Estimator comparison — {SUBJ_NC}, {ROI_NC}", fontsize=12)
plt.tight_layout()
plt.savefig("eeg_nc_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.4</strong><br>Which estimator is more likely to have been used to generate the stored EEG noise ceilings? Justify your answer with both visual and quantitative evidence.</div>The evidence overwhelmingly identifies the variance-based estimator as the one used to generate the stored ceilings. The scatter plot shows a perfect r = 1.0000 with MAE = 0.00% — the two are numerically identical to floating-point precision — while the split-half estimator achieves only r = 0.9312 with MAE = 4.84%. A paired t-test on the absolute errors confirms the difference is highly significant (t = −53.5, p ≈ 0), leaving no ambiguity. This makes methodological sense: the variance-based approach analytically decomposes z-scored variance into signal and noise components using all 80 repetitions simultaneously, producing a deterministic estimate that does not depend on a random partition. The split-half estimator is inherently stochastic and conservative by construction — averaging two 40-repetition halves captures less signal than using all 80, and the Spearman-Brown correction only exactly recovers the full-repetition ceiling under a strict linear noise model that the real data only approximately satisfies.

---

## 1.5 Convert NSD ncsnr to noise ceiling and visualize it on cortex

Some datasets, such as NSD, provide reliability estimates with the data release. In this section, you will visualize the provided NSD reliability estimates on the cortical surface and convert them into noise ceilings for later use in predictive analyses.

The provided NSD reliability estimates are stored as **ncsnr** values on the fsaverage surface. To use them as noise ceilings for voxel-wise analyses, you need to convert ncsnr to noise ceiling using the formula provided in the NSD methods paper.

Parcellations and atlases provide group-level anatomical labels for brain regions. They are often defined on a standard surface or volume space (e.g., fsaverage, MNI) and can be used to summarize or interpret neural data. For this exercise, use the Destrieux atlas to anatomically label the regions with the highest and lowest noise ceilings. It is available in fsaverage space and can be accessed through `nilearn`. Compute the average noise ceiling within each atlas region and identify which regions have the highest and lowest reliability.

If the available atlas is in a different surface resolution (e.g. `fsaverage5`), you can interpolate either the atlas or the noise ceiling map to the same space before visualization. Prefer downsampling rather than upsampling to avoid introducing artificial precision.

You can use `nibabel` to load the `.mgh` files and `nilearn` to visualize the resulting noise ceiling on the fsaverage surface.


<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Load the provided `.mgh` files for subject 01.
- Convert **ncsnr** to a **noise ceiling estimate** using the formula described in the NSD paper.
- Visualize the resulting noise ceiling on the fsaverage surface.
- Overlay a cortical parcellation.
- Compute parcel-wise average values.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One histogram of ncsnr values**
2. **One cortical surface plot** of ncsnr or the derived noise ceiling
3. **One cortical surface plot with parcel overlay**
4. **One parcel-wise summary figure or one parcel-wise summary table**


In [ ]:
# ============================================================
# 1.5  NSD ncsnr conversion and cortical visualization
# ============================================================
from nilearn import datasets, surface
from nilearn import plotting as nlplt

# --- Load surface ncsnr maps for subj01 ---
lh_img = nib.load(NSD_LH_PATH)
rh_img = nib.load(NSD_RH_PATH)

lh_ncsnr = lh_img.get_fdata().squeeze()   # (n_vertices_lh,)
rh_ncsnr = rh_img.get_fdata().squeeze()   # (n_vertices_rh,)

print(f"LH ncsnr: {lh_ncsnr.shape},  range [{lh_ncsnr.min():.3f}, {lh_ncsnr.max():.3f}]")
print(f"RH ncsnr: {rh_ncsnr.shape},  range [{rh_ncsnr.min():.3f}, {rh_ncsnr.max():.3f}]")

# --- Convert ncsnr to noise ceiling (explained variance) ---
# Formula (Allen et al. 2022 NSD paper):
#   NC = ncsnr^2 / (ncsnr^2 + 1/n_reps)
# For subj01 (full NSD): n_reps = 3 (some voxels may have fewer)
N_REPS = 3

def ncsnr_to_nc(ncsnr, n_reps=N_REPS):
    """Convert ncsnr to noise ceiling (proportion [0,1])."""
    snr2 = np.clip(ncsnr, 0, None) ** 2
    return snr2 / (snr2 + 1.0 / n_reps)

lh_nc = ncsnr_to_nc(lh_ncsnr)
rh_nc = ncsnr_to_nc(rh_ncsnr)

# --- Histogram of ncsnr ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(lh_ncsnr[lh_ncsnr > 0], bins=80, color="steelblue", alpha=0.7, label="LH")
axes[0].hist(rh_ncsnr[rh_ncsnr > 0], bins=80, color="tomato",    alpha=0.7, label="RH")
axes[0].set_xlabel("ncsnr"); axes[0].set_ylabel("Vertex count")
axes[0].set_title("NSD subj01 — ncsnr distribution")
axes[0].legend(); axes[0].spines[["top","right"]].set_visible(False)

axes[1].hist(lh_nc[lh_nc > 0], bins=80, color="steelblue", alpha=0.7, label="LH")
axes[1].hist(rh_nc[rh_nc > 0], bins=80, color="tomato",    alpha=0.7, label="RH")
axes[1].set_xlabel("Noise ceiling (EV, [0,1])"); axes[1].set_ylabel("Vertex count")
axes[1].set_title("NSD subj01 — noise ceiling distribution")
axes[1].legend(); axes[1].spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("nsd_ncsnr_hist.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Cortical surface plot of noise ceiling ---
fsaverage = datasets.fetch_surf_fsaverage(mesh="fsaverage")

fig = nlplt.plot_surf_stat_map(
    fsaverage["infl_left"], lh_nc,
    hemi="left", view="lateral",
    title="NSD subj01 — NC (LH, lateral)", colorbar=True,
    cmap="hot", vmax=0.7, bg_map=fsaverage["sulc_left"]
)
fig.savefig("nsd_nc_surface_lh.png", dpi=150, bbox_inches="tight")
plt.show()

fig = nlplt.plot_surf_stat_map(
    fsaverage["infl_right"], rh_nc,
    hemi="right", view="lateral",
    title="NSD subj01 — NC (RH, lateral)", colorbar=True,
    cmap="hot", vmax=0.7, bg_map=fsaverage["sulc_right"]
)
fig.savefig("nsd_nc_surface_rh.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Destrieux atlas parcellation ---
# Fetch at fsaverage5 then downsample ncsnr to match
destrieux = datasets.fetch_atlas_surf_destrieux()
# labels are on fsaverage5 (10242 vertices per hemi)
# Downsample our ncsnr (163842 vertices) to fsaverage5 to match atlas resolution
fsaverage5 = datasets.fetch_surf_fsaverage(mesh="fsaverage5")

lh_nc_ds = surface.vol_to_surf(
    nib.Nifti1Image(lh_nc[:, np.newaxis, np.newaxis],
                    np.eye(4)),   # dummy volume — we use direct approach below
    fsaverage5["pial_left"]
) if False else None  # placeholder

# Cleaner approach: use nilearn's resample_to_img on surface
# Since we have the data as 1-D vertex arrays, we resample via nearest-vertex
from nilearn.surface import load_surf_data

# Load parcellation labels (fsaverage5, 10242 vertices)
lh_labels_fs5 = destrieux["map_left"]   # (10242,) int array of parcel IDs
rh_labels_fs5 = destrieux["map_right"]

# Downsample nc map from fsaverage (163842) to fsaverage5 (10242)
# by sampling the nc value at each fsaverage5 vertex's nearest fsaverage vertex.
# nilearn provides coords for both meshes:
coords_fs_lh  = surface.load_surf_mesh(fsaverage["pial_left"])[0]    # (163842, 3)
coords_fs5_lh = surface.load_surf_mesh(fsaverage5["pial_left"])[0]   # (10242, 3)

from sklearn.neighbors import KDTree
kdt_lh = KDTree(coords_fs_lh)
_, nn_lh = kdt_lh.query(coords_fs5_lh, k=1)
nn_lh = nn_lh.squeeze()
lh_nc_fs5 = lh_nc[nn_lh]   # (10242,) downsampled nc

coords_fs_rh  = surface.load_surf_mesh(fsaverage["pial_right"])[0]
coords_fs5_rh = surface.load_surf_mesh(fsaverage5["pial_right"])[0]
kdt_rh = KDTree(coords_fs_rh)
_, nn_rh = kdt_rh.query(coords_fs5_rh, k=1)
nn_rh = nn_rh.squeeze()
rh_nc_fs5 = rh_nc[nn_rh]

# --- Parcel-wise mean NC ---
parcel_ids_lh = np.unique(lh_labels_fs5)
parcel_names  = destrieux["labels"]   # list of parcel name strings

rows_parcel = []
for pid in parcel_ids_lh:
    if pid == 0:
        continue
    mask   = lh_labels_fs5 == pid
    nc_val = lh_nc_fs5[mask].mean() if mask.sum() > 0 else np.nan
    name   = parcel_names[pid].decode() if isinstance(parcel_names[pid], bytes) \
             else str(parcel_names[pid])
    rows_parcel.append({"Parcel ID": pid, "Parcel name": name,
                        "Mean NC (LH)": nc_val, "N vertices": mask.sum()})

parcel_df = (pd.DataFrame(rows_parcel)
               .sort_values("Mean NC (LH)", ascending=False)
               .reset_index(drop=True))

print("\nTop-10 most reliable parcels (LH):")
print(parcel_df.head(10)[["Parcel name","Mean NC (LH)","N vertices"]].to_string(index=False))
print("\nBottom-10 least reliable parcels (LH):")
print(parcel_df.tail(10)[["Parcel name","Mean NC (LH)","N vertices"]].to_string(index=False))

# --- Parcel-wise bar chart (top + bottom 10) ---
top_bot = pd.concat([parcel_df.head(10), parcel_df.tail(10)])
fig, ax = plt.subplots(figsize=(10, 6))
colors  = ["steelblue"] * 10 + ["tomato"] * 10
ax.barh(top_bot["Parcel name"], top_bot["Mean NC (LH)"], color=colors)
ax.axvline(parcel_df["Mean NC (LH)"].median(), color="black", ls="--",
           lw=1, label="Median NC")
ax.set_xlabel("Mean noise ceiling [0,1]")
ax.set_title("NSD subj01 — parcel-wise NC (Destrieux atlas, LH)\n"
             "Blue = top 10, Red = bottom 10")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("nsd_parcelwise_nc.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Surface plot with parcel overlay ---
fig = nlplt.plot_surf_roi(
    fsaverage5["infl_left"],
    roi_map=lh_labels_fs5,
    hemi="left", view="lateral",
    bg_map=fsaverage5["sulc_left"],
    title="NSD subj01 — Destrieux parcels (LH)",
    colorbar=False
)
fig.savefig("nsd_parcel_overlay.png", dpi=150, bbox_inches="tight")
plt.show()


<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.5</strong><br>Which cortical regions appear most reliable, and which appear least reliable? Explain how the parcellation helps interpret the surface maps.</div>

The cortical reliability map shows a clear occipital-to-frontal gradient: the most reliable parcels are entirely concentrated in visual cortex, with the top-10 dominated by occipital sulci and gyri (middle occipital gyrus, occipital pole, collateral transverse sulcus, lingual gyrus, and fusiform cortex), all with mean noise ceilings above 0.29. This is exactly what one would expect from a passive visual experiment — these regions are strongly and consistently driven by naturalistic images, so their BOLD responses are reproducible across trials. The fusiform entry (G_oc-temp_lat-fusifor, NC ≈ 0.29) is particularly noteworthy as it corresponds to the ventral object-selective stream, confirming that higher visual areas are also reliably activated. By contrast, the least reliable parcels are entirely non-visual: somatosensory and motor cortex (postcentral gyrus, central sulcus, paracentral gyrus), temporal pole, subcallosal cortex, and medial wall structures, all with mean noise ceilings below 0.03 — these areas respond weakly and inconsistently to passive image viewing relative to their trial-by-trial noise. The Destrieux parcellation is indispensable for this interpretation: the raw vertex-level surface map is difficult to read anatomically, but collapsing to named parcels immediately reveals that the reliability gradient follows a known visual hierarchy and that non-visual association cortex is essentially at noise floor, which both validates the preprocessing and defines which ROIs are meaningful targets for brain–model alignment in later sections.

---

# 2. Brain–Model Alignment

In this section, you will compare neural responses and model features using **both representational metrics and predictive linear models**. You must complete both parts of this section. The goal is not only to report scores, but also to compare what different metrics reveal about model–brain alignment.

## 2.1 Representational alignment: RSA

RSA stands for representational similarity analysis. It is one of the most widely used analyses in fMRI and model–brain alignment research. It compares the geometry of two representational spaces through their representational dissimilarity matrices (RDMs). Given two response matrices, `X` and `Y`, with rows corresponding to the same stimuli, we first compute an RDM for each matrix using correlation distance:

$$
D^X_{ij} = 1 - \mathrm{corr}(X[i,:], X[j,:]),
\qquad
D^Y_{ij} = 1 - \mathrm{corr}(Y[i,:], Y[j,:]),
$$

for stimulus pairs $i \neq j$.

We then vectorize the upper triangle of each RDM and compute RSA as the Spearman correlation between these two vectors:

$$
\mathrm{RSA}(X, Y)
=
\rho_{\mathrm{Spearman}}
\left(
\mathrm{vec}(D^X),\,
\mathrm{vec}(D^Y)
\right).
$$

In this project, `X` will usually denote model features from one candidate layer, and `Y` will denote neural responses from one dataset, ROI, subject, or time slice, depending on the analysis.

- Implement RSA between two representation matrices.
- Support at least one dissimilarity measure and one similarity measure.
- Use your implementation to compare model layers to neural responses.

In [15]:
from typing import Literal
import numpy as np
from scipy import stats
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

class RepresentationalSimilarityAnalysis:
    """
    Representational Similarity Analysis (RSA).

    Computes RDMs from two representation matrices and correlates
    their upper triangles (Spearman by default, as is standard in RSA).
    """

    def __init__(
        self,
        dissimilarity: Literal["correlation", "euclidean", "cosine"] = "correlation",
        similarity_metric: Literal["pearson", "spearman"] = "spearman",
    ):
        self.dissimilarity = dissimilarity
        self.similarity_metric = similarity_metric

    def __call__(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Compute RSA similarity between X and Y.

        Parameters
        ----------
        X, Y : np.ndarray
            Arrays of shape (n_conditions, ...) that may need to be flattened
            along feature dimensions.

        Returns
        -------
        rsa_similarity : float
            Correlation between the vectorized upper triangles of the two RDMs.
        """
        return self.forward(X, Y)

    def forward(self, X: np.ndarray, Y: np.ndarray) -> float:
        X = np.asarray(X, dtype=np.float64).reshape(X.shape[0], -1)
        Y = np.asarray(Y, dtype=np.float64).reshape(Y.shape[0], -1)
        rdm_x = self.compute_rdm(X)
        rdm_y = self.compute_rdm(Y)
        return self.compare_rdms(rdm_x, rdm_y)

    def compute_rdm(self, X: np.ndarray) -> np.ndarray:
        """
        Compute the Representational Dissimilarity Matrix (RDM)
        for a given representation matrix X.

        Parameters
        ----------
        X : np.ndarray
            Array of shape (n_conditions, n_features).

        Returns
        -------
        rdm : np.ndarray
            Array of shape (n_conditions, n_conditions) with pairwise dissimilarities.
        """
        X = np.asarray(X, dtype=np.float64)
        if self.dissimilarity == "correlation":
            # Efficient vectorised correlation distance
            X_c = X - X.mean(axis=1, keepdims=True)
            norms = np.linalg.norm(X_c, axis=1, keepdims=True)
            norms = np.where(norms == 0, 1.0, norms)
            X_n = X_c / norms
            rdm = 1.0 - X_n @ X_n.T
            np.fill_diagonal(rdm, 0.0)
        elif self.dissimilarity == "cosine":
            rdm = cosine_distances(X)
        elif self.dissimilarity == "euclidean":
            rdm = euclidean_distances(X)
        else:
            raise ValueError(f"Unknown dissimilarity: {self.dissimilarity}")
        return rdm

    def compare_rdms(self, rdm1: np.ndarray, rdm2: np.ndarray) -> float:
        """
        Compare two RDMs by correlating their upper triangles.
        """
        n = rdm1.shape[0]
        idx = np.triu_indices(n, k=1)
        v1 = rdm1[idx]
        v2 = rdm2[idx]
        if self.similarity_metric == "spearman":
            r, _ = stats.spearmanr(v1, v2)
        else:
            r, _ = stats.pearsonr(v1, v2)
        return float(r)


## 2.2 Representational alignment: unbiased linear CKA

CKA stands for centered kernel alignment. It is commonly used in interpretability and representation analysis to test how strongly the internal computations of two systems align. As a second mapping-free alignment metric, we want to compute unbiased linear centered kernel alignment (CKA) between model features and neural responses. Let

$$
X \in \mathbb{R}^{n \times d}, \qquad
Y \in \mathbb{R}^{n \times p},
$$

where both matrices are measured on the same $n$ stimuli. We form linear Gram matrices

$$
K = XX^\top,
\qquad
L = YY^\top.
$$

We then estimate dependence using the unbiased (U-statistic) HSIC estimator, $\mathrm{HSIC}_u(K, L)$, and define CKA as

$$
\mathrm{CKA}(X, Y)
=
\frac{\mathrm{HSIC}_u(K, L)}
{\sqrt{\mathrm{HSIC}_u(K, K)\,\mathrm{HSIC}_u(L, L)}}.
$$

Like RSA, CKA compares representational structure directly without fitting a predictive mapping. In this notebook, `X` and `Y` again refer to aligned model and neural response matrices evaluated on the same set of stimuli.

- Implement **unbiased linear CKA** only.
- Use your implementation to compare model layers to neural responses.

In [16]:
import numpy as np

class CenteredKernelAlignment:
    """
    Unbiased linear CKA only.

    Parameters
    ----------
    eps : float
        Small constant for numerical stability.
    dtype : np.dtype
        Data type used for computations.
    """

    def __init__(
        self,
        eps: float = 1e-8,
        dtype: np.dtype = np.float64,
    ):
        self.eps = eps
        self.dtype = dtype

    def __call__(self, X: np.ndarray, Y: np.ndarray) -> float:
        return self.forward(X, Y)

    def forward(self, X: np.ndarray, Y: np.ndarray) -> float:
        X = np.asarray(X).astype(self.dtype)
        Y = np.asarray(Y).astype(self.dtype)
        
        if X.shape[0] != Y.shape[0]:
            raise ValueError(
                f"Batch sizes must match along axis 0: {X.shape[0]} vs {Y.shape[0]}"
            )

        # Flatten to (n_samples, n_features)
        X = X.reshape(X.shape[0], -1)
        Y = Y.reshape(Y.shape[0], -1)
        
        return self._unbiased_linear_cka(X, Y)

    def _unbiased_linear_hsic(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Unbiased HSIC estimator for the linear kernel.

        X : [n, d_x]
        Y : [n, d_y]
        """
        n = X.shape[0]
        # Gram matrices
        K = X @ X.T
        L = Y @ Y.T
        # Zero diagonals (K_tilde, L_tilde)
        np.fill_diagonal(K, 0.0)
        np.fill_diagonal(L, 0.0)
        # Scalar sums
        sum_K = K.sum()
        sum_L = L.sum()
        sum_KL = (K * L).sum()  # trace(K @ L) for symmetric = element-wise
        # Actually for HSIC_u we need trace(K_tilde @ L_tilde) - use einsum
        tr_KL = np.einsum('ij,ji->', K, L)
        ones_K = K.sum(axis=1)   # shape (n,)
        ones_L = L.sum(axis=1)
        dot_KL_1 = ones_K @ ones_L  # 1^T K L 1
        hsic = (
            tr_KL
            + (sum_K * sum_L) / ((n - 1) * (n - 2))
            - 2.0 * dot_KL_1 / (n - 2)
        ) / (n * (n - 3))
        return float(hsic)

    def _unbiased_linear_cka(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Unbiased linear CKA:

            CKA_unb(X, Y) =
                HSIC_unb(X, Y) / sqrt(HSIC_unb(X, X) * HSIC_unb(Y, Y))
        """
        hsic_xy = self._unbiased_linear_hsic(X, Y)
        hsic_xx = self._unbiased_linear_hsic(X, X)
        hsic_yy = self._unbiased_linear_hsic(Y, Y)
        denom = np.sqrt(max(hsic_xx, 0.0) * max(hsic_yy, 0.0))
        if denom < self.eps:
            return 0.0
        return float(hsic_xy / denom)


## 2.3 Apply RSA and CKA

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Compare layers within each model.
- Compare the two models.
- For EEG, show how representational similarity changes over time.
- For TVSD and NSD, compare across ROIs.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Layer-wise RSA results** for both models.
2. **Layer-wise CKA results** for both models.
3. **One direct comparison between the two models** using representational metrics.
4. **One EEG time-resolved analysis** or **one ROI-wise analysis** for TVSD/NSD.
5. **One short written interpretation** in Answer box 2.1. (below code cells)

In [ ]:
# ============================================================
# 2.3  Apply RSA and CKA across layers, models, datasets
# ============================================================

import gc
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ────────────────────────────────────────────────────────────
# Metrics
# ────────────────────────────────────────────────────────────
rsa_fn = RepresentationalSimilarityAnalysis(
    dissimilarity="correlation",
    similarity_metric="spearman"
)
cka_fn = CenteredKernelAlignment()

# ────────────────────────────────────────────────────────────
# 🔧 NEW: robust dataset resolver (fixes your bug)
# ────────────────────────────────────────────────────────────
def get_dataset(f, key):
    obj = f[key]
    if isinstance(obj, h5py.Dataset):
        return obj
    elif isinstance(obj, h5py.Group):
        # Try common naming first
        for name in ["features", "data", "activations"]:
            if name in obj:
                return obj[name]
        # fallback: first dataset found
        for v in obj.values():
            if isinstance(v, h5py.Dataset):
                return v
        raise ValueError(f"No dataset found inside group '{key}'")
    else:
        raise TypeError(f"Unsupported HDF5 object: {type(obj)}")

# ────────────────────────────────────────────────────────────
# Helper functions
# ────────────────────────────────────────────────────────────
def load_feat_rows(f, layer_key, idx_sorted, idx_restore):
    dset = get_dataset(f, layer_key)
    data = dset[idx_sorted]
    return data[idx_restore].astype(np.float32)

def subsample_features(feat, n_feat=1000, seed=0):
    rng = np.random.default_rng(seed)
    if feat.shape[1] > n_feat:
        idx = rng.choice(feat.shape[1], n_feat, replace=False)
        return feat[:, idx]
    return feat

def sorted_indices(idx):
    order = np.argsort(idx)
    restore = np.argsort(order)
    return idx[order], restore

# ────────────────────────────────────────────────────────────
# 🔧 NEW: consistent layer key extraction
# ────────────────────────────────────────────────────────────
def get_layer_keys(f):
    keys = []
    for k in f.keys():
        if k == "ids":
            continue
        obj = f[k]
        if isinstance(obj, h5py.Dataset):
            keys.append(k)
        elif isinstance(obj, h5py.Group):
            for subk in obj.keys():
                keys.append(f"{k}/{subk}")
    return sorted(keys)

# ────────────────────────────────────────────────────────────
# 1. TVSD
# ────────────────────────────────────────────────────────────
print("=" * 60)
print("TVSD — RSA / CKA layer profiles")
print("=" * 60)

tvsd_results = []

tvsd_test_si_A, tvsd_test_ri_A = sorted_indices(tvsd_test_feat_idx_A)
tvsd_test_si_B, tvsd_test_ri_B = sorted_indices(tvsd_test_feat_idx_B)

with h5py.File(TVSD_PATH, "r") as fneural, \
     h5py.File(FEAT_A_THINGS, "r") as fA, \
     h5py.File(FEAT_B_THINGS, "r") as fB:

    layers_A = get_layer_keys(fA)
    layers_B = get_layer_keys(fB)

    for monkey in ["monkeyF", "monkeyN"]:
        for roi in ["V1", "V4", "IT"]:
            Y = fneural[f"test/neural_data/{monkey}/{roi}"][:].astype(np.float32)
            Y_sub = subsample_features(Y)

            for model_label, ffeat, layers, si, ri in [
                ("ResNet152-adv", fA, layers_A, tvsd_test_si_A, tvsd_test_ri_A),
                ("Qwen3-VL-2B",   fB, layers_B, tvsd_test_si_B, tvsd_test_ri_B),
            ]:
                for li, lk in enumerate(layers):
                    X = load_feat_rows(ffeat, lk, si, ri)
                    X_sub = subsample_features(X)

                    rsa_val = rsa_fn(X_sub, Y_sub)
                    cka_val = cka_fn(X_sub, Y_sub)

                    tvsd_results.append({
                        "dataset": "TVSD",
                        "monkey": monkey,
                        "roi": roi,
                        "model": model_label,
                        "layer_idx": li,
                        "layer": lk,
                        "RSA": rsa_val,
                        "CKA": cka_val,
                    })

                    del X, X_sub
                gc.collect()

            del Y, Y_sub

tvsd_df = pd.DataFrame(tvsd_results)
print(tvsd_df.groupby(["model", "roi"])[["RSA", "CKA"]].max().round(4))


# ────────────────────────────────────────────────────────────
# 2. EEG
# ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("EEG — time-resolved RSA / CKA")
print("=" * 60)

SUBJ_EEG = "sub-01"
ROI_EEG  = "occipital_parietal"
TIME     = np.linspace(0, 0.8, 80)

eeg_time_results = []

eeg_test_si_A, eeg_test_ri_A = sorted_indices(eeg_test_feat_idx_A)
eeg_test_si_B, eeg_test_ri_B = sorted_indices(eeg_test_feat_idx_B)

with h5py.File(EEG_PATH, "r") as fneural, \
     h5py.File(FEAT_A_THINGS, "r") as fA, \
     h5py.File(FEAT_B_THINGS, "r") as fB:

    Y_eeg = fneural[f"test/neural_data/{SUBJ_EEG}/{ROI_EEG}"][:]

    layers_A = get_layer_keys(fA)
    layers_B = get_layer_keys(fB)

    n_tp_eeg = Y_eeg.shape[2]

    for model_label, ffeat, layers, si, ri in [
        ("ResNet152-adv", fA, layers_A, eeg_test_si_A, eeg_test_ri_A),
        ("Qwen3-VL-2B",   fB, layers_B, eeg_test_si_B, eeg_test_ri_B),
    ]:
        for li, lk in enumerate(layers):
            X = load_feat_rows(ffeat, lk, si, ri)
            X_sub = subsample_features(X)

            for tp in range(n_tp_eeg):
                Y_tp = Y_eeg[:, :, tp]
                Y_sub = subsample_features(Y_tp)

                rsa_val = rsa_fn(X_sub, Y_sub)
                cka_val = cka_fn(X_sub, Y_sub)

                eeg_time_results.append({
                    "model": model_label,
                    "layer_idx": li,
                    "layer": lk,
                    "time": TIME[tp],
                    "RSA": rsa_val,
                    "CKA": cka_val,
                })

            del X, X_sub
        gc.collect()

eeg_time_df = pd.DataFrame(eeg_time_results)


# ────────────────────────────────────────────────────────────
# 3. NSD
# ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("NSD — ROI-wise RSA / CKA (subj01)")
print("=" * 60)

NSD_ROIS = ["V1v","V2v","V3v","hV4","FFA-1","VWFA-1","PPA","OPA","EBA"]
SUBJ_NSD = "subj01"

nsd_results = []

nsd_test_si_A, nsd_test_ri_A = sorted_indices(nsd_test_feat_idx_A[SUBJ_NSD])
nsd_test_si_B, nsd_test_ri_B = sorted_indices(nsd_test_feat_idx_B[SUBJ_NSD])

with h5py.File(NSD_PATH, "r") as fneural, \
     h5py.File(FEAT_A_NSD, "r") as fA, \
     h5py.File(FEAT_B_NSD, "r") as fB:

    layers_A = get_layer_keys(fA)
    layers_B = get_layer_keys(fB)

    for roi in NSD_ROIS:
        try:
            Y = fneural[f"test/neural_data/{SUBJ_NSD}/{roi}"][:].astype(np.float32)
        except KeyError:
            continue

        Y_sub = subsample_features(Y)

        for model_label, ffeat, layers, si, ri in [
            ("ResNet152-adv", fA, layers_A, nsd_test_si_A, nsd_test_ri_A),
            ("Qwen3-VL-2B",   fB, layers_B, nsd_test_si_B, nsd_test_ri_B),
        ]:
            for li, lk in enumerate(layers):
                X = load_feat_rows(ffeat, lk, si, ri)
                X_sub = subsample_features(X)

                rsa_val = rsa_fn(X_sub, Y_sub)
                cka_val = cka_fn(X_sub, Y_sub)

                nsd_results.append({
                    "dataset": "NSD",
                    "subj": SUBJ_NSD,
                    "roi": roi,
                    "model": model_label,
                    "layer_idx": li,
                    "layer": lk,
                    "RSA": rsa_val,
                    "CKA": cka_val,
                })

                del X, X_sub
            gc.collect()

        del Y, Y_sub

nsd_rep_df = pd.DataFrame(nsd_results)
print(nsd_rep_df.groupby(["model","roi"])[["RSA","CKA"]].max().round(4))


# ────────────────────────────────────────────────────────────
# Layer keys (for inspection)
# ────────────────────────────────────────────────────────────
print("\nLayer keys Model A:", layers_A)
print("Layer keys Model B:", layers_B)


In [ ]:
# ============================================================
# 2.3  Plots: layer-wise RSA and CKA
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=False)

# ── TVSD: RSA and CKA per model, per ROI ────────────────────
ax = axes[0][0]
for model, ls in [("ResNet152-adv","--"), ("Qwen3-VL-2B","-")]:
    for roi, color in [("V1","steelblue"),("V4","goldenrod"),("IT","tomato")]:
        sub = tvsd_df[(tvsd_df.model == model) & (tvsd_df.roi == roi)
                      & (tvsd_df.monkey == "monkeyF")].sort_values("layer_idx")
        ax.plot(sub["layer_idx"], sub["RSA"], color=color, ls=ls, lw=2,
                label=f"{model[:6]} {roi}" if roi == "V1" else None)
ax.set_title("TVSD — RSA (monkeyF)"); ax.set_xlabel("Layer"); ax.set_ylabel("RSA")
# Custom legend
from matplotlib.lines import Line2D
handles = [
    Line2D([0],[0], color="steelblue", lw=2, label="V1"),
    Line2D([0],[0], color="goldenrod", lw=2, label="V4"),
    Line2D([0],[0], color="tomato",    lw=2, label="IT"),
    Line2D([0],[0], color="gray", lw=2, ls="--", label="ResNet152"),
    Line2D([0],[0], color="gray", lw=2, ls="-",  label="Qwen3-VL"),
]
ax.legend(handles=handles, fontsize=7); ax.spines[["top","right"]].set_visible(False)

ax = axes[1][0]
for model, ls in [("ResNet152-adv","--"), ("Qwen3-VL-2B","-")]:
    for roi, color in [("V1","steelblue"),("V4","goldenrod"),("IT","tomato")]:
        sub = tvsd_df[(tvsd_df.model == model) & (tvsd_df.roi == roi)
                      & (tvsd_df.monkey == "monkeyF")].sort_values("layer_idx")
        ax.plot(sub["layer_idx"], sub["CKA"], color=color, ls=ls, lw=2)
ax.set_title("TVSD — CKA (monkeyF)"); ax.set_xlabel("Layer"); ax.set_ylabel("CKA")
ax.spines[["top","right"]].set_visible(False)

# ── EEG: time-resolved RSA / CKA (best layer) ───────────────
ax = axes[0][1]
for model, ls, color in [("ResNet152-adv","--","steelblue"),
                          ("Qwen3-VL-2B","-","tomato")]:
    # Best layer per time point
    sub = eeg_time_df[eeg_time_df.model == model]
    best = sub.groupby("time")["RSA"].max().reset_index()
    ax.plot(best["time"], best["RSA"], color=color, ls=ls, lw=2, label=model[:10])
ax.set_title("EEG — RSA (best layer, per time)"); ax.set_xlabel("Time (s)"); ax.set_ylabel("RSA")
ax.legend(fontsize=7); ax.spines[["top","right"]].set_visible(False)

ax = axes[1][1]
for model, ls, color in [("ResNet152-adv","--","steelblue"),
                          ("Qwen3-VL-2B","-","tomato")]:
    sub = eeg_time_df[eeg_time_df.model == model]
    best = sub.groupby("time")["CKA"].max().reset_index()
    ax.plot(best["time"], best["CKA"], color=color, ls=ls, lw=2)
ax.set_title("EEG — CKA (best layer, per time)"); ax.set_xlabel("Time (s)"); ax.set_ylabel("CKA")
ax.spines[["top","right"]].set_visible(False)

# ── NSD: ROI-wise best RSA / CKA ────────────────────────────
ax = axes[0][2]
roi_order = ["V1v","V2v","V3v","hV4","FFA-1","VWFA-1","PPA","OPA","EBA"]
roi_order = [r for r in roi_order if r in nsd_rep_df["roi"].unique()]
x = np.arange(len(roi_order))
w = 0.2
for i, (model, color) in enumerate([("ResNet152-adv","steelblue"),("Qwen3-VL-2B","tomato")]):
    vals = [nsd_rep_df[(nsd_rep_df.model==model) & (nsd_rep_df.roi==r)]["RSA"].max()
            for r in roi_order]
    ax.bar(x + i*w, vals, w, color=color, alpha=0.8, label=model[:10])
ax.set_xticks(x + w/2); ax.set_xticklabels(roi_order, rotation=45, ha="right", fontsize=7)
ax.set_title("NSD — best RSA per ROI"); ax.set_ylabel("RSA")
ax.legend(fontsize=7); ax.spines[["top","right"]].set_visible(False)

ax = axes[1][2]
for i, (model, color) in enumerate([("ResNet152-adv","steelblue"),("Qwen3-VL-2B","tomato")]):
    vals = [nsd_rep_df[(nsd_rep_df.model==model) & (nsd_rep_df.roi==r)]["CKA"].max()
            for r in roi_order]
    ax.bar(x + i*w, vals, w, color=color, alpha=0.8)
ax.set_xticks(x + w/2); ax.set_xticklabels(roi_order, rotation=45, ha="right", fontsize=7)
ax.set_title("NSD — best CKA per ROI"); ax.set_ylabel("CKA")
ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Layer-wise RSA and CKA — all datasets", fontsize=13)
plt.tight_layout()
plt.savefig("sec2_rsa_cka_overview.png", dpi=150, bbox_inches="tight")
plt.show()


<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.1</strong><br>Do RSA and CKA tell the same story? Identify at least one case where they agree and one case where they disagree, and explain what that might mean.</div>

RSA and CKA largely tell a consistent story, but with some important differences. They agree, for instance, in the TVSD dataset: for ResNet152-adv, both metrics identify IT and V4 as the most alignable ROIs and consistently rank this model above Qwen3-VL-2B. In NSD, they also agree that early visual areas (V1v, V2v) show the highest alignment, with similar layer-wise trends.

A clear disagreement appears for V1 in TVSD. For ResNet152-adv, CKA (0.33) is much higher than RSA (0.17), while for Qwen3-VL-2B the two are closer. This suggests that ResNet’s alignment with V1 is driven by a few dominant dimensions (captured by CKA), whereas the overall pairwise geometry is weaker (captured by RSA). In general, when they diverge, it indicates that shared representations are concentrated in a low-dimensional, anisotropic structure rather than spread across the full representational space.

---

## 2.4 Predictive alignment: linear encoding models

## Linear encoding model

In the predictive part of the project, you will map model features to neural responses using a **linear encoding model with L2 regularization (ridge regression)**.

For a stimulus $\mathbf{x}$, let $\mathbf{z}_{\ell}(\mathbf{x})$ denote the feature vector extracted from model layer $\ell$. For a given subject $s$ and neural target $r$ (for example an ROI, a group of voxels, or a set of channels / time points), the predicted neural response is

$$
\widehat{\mathbf{y}}_{r,s}(\mathbf{x})
=
W_{r,s}\,\mathbf{z}_{\ell}(\mathbf{x}) + \mathbf{b}_{r,s},
$$

where:
- $\mathbf{z}_{\ell}(\mathbf{x}) \in \mathbb{R}^{d}$ is the model feature vector,
- $\widehat{\mathbf{y}}_{r,s}(\mathbf{x}) \in \mathbb{R}^{p}$ is the predicted neural response,
- $W_{r,s} \in \mathbb{R}^{p \times d}$ is the learned linear mapping,
- $\mathbf{b}_{r,s} \in \mathbb{R}^{p}$ is a bias term.

We fit the mapping on the training split using ridge regression:

$$
\min_{W_{r,s},\,\mathbf{b}_{r,s}}
\sum_{\mathbf{x}\in\mathcal{D}_{\mathrm{train}}}
\left\|
\mathbf{y}_{r,s}(\mathbf{x}) - \widehat{\mathbf{y}}_{r,s}(\mathbf{x})
\right\|_2^2
\;+\;
\alpha \left\|W_{r,s}\right\|_F^2.
$$

Here, $\mathbf{y}_{r,s}(\mathbf{x})$ is the measured neural response, and $\alpha$ controls the strength of L2 regularization. Larger $\alpha$ penalizes large weights more strongly, which can improve generalization when the feature dimension is high. You should select $\alpha$ using only the training data, for example with a validation split or cross-validation, and then evaluate the final model on the held-out test set.

<div style="background:#eef8f4; border-left:4px solid #5b9a7a; padding:8px 12px; border-radius:6px; font-weight:700; color:#285943;">Select the required targets</div>

Use the following targets:

- **TVSD:** all ROIs
- **EEG2:** `occipital_parietal`
- **NSD:** `V1v`, `V2v`, `V3v`, `hV4`, `FFA-1`, `VWFA-1`, `PPA`, `OPA`, `EBA`

You may explore additional ROIs if you wish.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.2</strong><br>Briefly explain why these targets are scientifically interesting. Are they chosen mainly for reliability, interpretability, or both?</div>

The selected targets cover key stages of the visual processing hierarchy and offer complementary views on brain–model alignment. TVSD areas V1, V4, and IT span the macaque ventral stream from low-level edge processing to object-selective representations, making them ideal for testing whether model depth matches neural depth. The EEG occipital-parietal region captures early and mid-level visual activity with high temporal resolution and reliability, improving interpretability. NSD ROIs (V1v through EBA) include both early visual and category-selective areas (e.g., FFA-1, PPA, EBA, VWFA-1), enabling tests of hierarchical alignment. Overall, these targets are chosen for their strong signal quality and well-understood functional roles.


In [ ]:
# ============================================================
# 2.4  Define required targets and load neural test data
# ============================================================

import h5py
import numpy as np

# TVSD targets
TVSD_MONKEYS = ["monkeyF", "monkeyN"]
TVSD_ROIS    = ["V1", "V4", "IT"]

# EEG2 target
EEG_SUBJ     = "sub-01"
EEG_ROI      = "occipital_parietal"

# NSD targets
NSD_ROIS_TARGET = ["V1v","V2v","V3v","hV4","FFA-1","VWFA-1","PPA","OPA","EBA"]
NSD_SUBJ        = "subj01"   # compute for subj01; structure same for others

# Quick check — print available voxel counts per NSD ROI
print("NSD ROI voxel counts (subj01, test split):")
with h5py.File(NSD_PATH, "r") as f:
    for roi in NSD_ROIS_TARGET:
        try:
            shape = f[f"test/neural_data/{NSD_SUBJ}/{roi}"].shape
            print(f"  {roi:12s}: {shape}")
        except KeyError:
            print(f"  {roi:12s}: NOT FOUND")

print("\nEEG test neural data shape:")
with h5py.File(EEG_PATH, "r") as f:
    shape = f[f"test/neural_data/{EEG_SUBJ}/{EEG_ROI}"].shape
    print(f"  {EEG_ROI}: {shape}")

print("\nTVSD test neural data shapes:")
with h5py.File(TVSD_PATH, "r") as f:
    for m in TVSD_MONKEYS:
        for r in TVSD_ROIS:
            print(f"  {m}/{r}: {f[f'test/neural_data/{m}/{r}'].shape}")


<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

For each dataset, target, model, and candidate layer:

- fit a **linear encoding model**,
- select hyperparameters without using the test split,
- evaluate on the test split.

Use iterative solvers (e.g. SGD, Adam) when needed to avoid memory issues, since `sklearn` Ridge might cause OOM.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **A clearly defined train/validation/test procedure** that does not use the test set for model selection.
2. **Linear encoding model results** for all required datasets and targets.
3. **The following predictive metrics:** Pearson correlation, noise-corrected Pearson correlation, explained variance, and noise-corrected explained variance.
4. **The following hybrid representational metrics on predicted responses:** encoding-RSA and encoding-CKA.
5. **Layer-wise plots** showing performance across candidate layers.
6. **One best-layer summary table** for the required targets.
7. **One comparison between the two models** using predictive results.
8. **One short written interpretation** in Answer box 2.3. (below code cell)

In [ ]:
# ============================================================
# PCA DIMENSION BENCHMARK
# One dataset (TVSD), one monkey, one ROI, first N_LAYERS layers
# ============================================================

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ── 1. Ridge + PCA ───────────────────────────────────────────

def fit_ridge_encoding(X_train, Y_train, X_test, n_components=512):
    X_train = X_train.astype(np.float32)
    X_test  = X_test.astype(np.float32)
    Y_train = Y_train.astype(np.float32)

    n_comp = min(n_components, X_train.shape[0] - 1, X_train.shape[1])
    pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=0)
    pca.fit(X_train)

    Xtr_p = pca.transform(X_train)
    Xte_p = pca.transform(X_test)
    del X_train, X_test

    sc = StandardScaler()
    Xtr_p = sc.fit_transform(Xtr_p)
    Xte_p = sc.transform(Xte_p)

    ridge = RidgeCV(alphas=ALPHA_GRID, cv=5, scoring="r2")
    ridge.fit(Xtr_p, Y_train)

    return ridge.predict(Xte_p).astype(np.float32)

# ── 2. Helpers ───────────────────────────────────────────────

def _sorted(idx):
    o = np.argsort(idx)
    return idx[o], np.argsort(o)

def load_tvsd_nc(monkey, roi):
    with h5py.File(TVSD_PATH, "r") as f:
        return f[f"noise_ceilings/{monkey}/{roi}"][:] / 100.0

def get_dataset(f, key):
    obj = f[key]

    if isinstance(obj, h5py.Dataset):
        return obj

    elif isinstance(obj, h5py.Group):
        for name in ["features", "data", "activations"]:
            if name in obj:
                return obj[name]

        for v in obj.values():
            if isinstance(v, h5py.Dataset):
                return v

        raise ValueError(f"No dataset found inside group '{key}'")

    else:
        raise TypeError(f"Unsupported object type: {type(obj)}")

def load_feat(f, layer_key, idx_sorted, idx_restore):
    dset = get_dataset(f, layer_key)
    data = dset[idx_sorted]
    return data[idx_restore].astype(np.float32)

def pearsonr_cols(Y_true, Y_pred):
    yt = Y_true - Y_true.mean(axis=0, keepdims=True)
    yp = Y_pred - Y_pred.mean(axis=0, keepdims=True)
    num = (yt * yp).sum(axis=0)
    den = np.sqrt((yt**2).sum(axis=0) * (yp**2).sum(axis=0))
    den = np.where(den < 1e-12, np.nan, den)
    return num / den

def ev_cols(Y_true, Y_pred):
    ss_res = ((Y_true - Y_pred)**2).sum(axis=0)
    ss_tot = ((Y_true - Y_true.mean(axis=0))**2).sum(axis=0)
    ss_tot = np.where(ss_tot < 1e-12, np.nan, ss_tot)
    return 1.0 - ss_res / ss_tot

def nc_correct_r(r, nc):
    return r / np.sqrt(np.clip(nc, 1e-6, None))

def nc_correct_ev(ev, nc):
    return ev / np.clip(nc, 1e-6, None)

# ── 3. Settings ──────────────────────────────────────────────

PCA_LIST     = [128, 256, 512, 1024, 2048]
BENCH_MONKEY = "monkeyF"
BENCH_ROI    = "IT"
N_LAYERS     = 5
ALPHA_GRID = np.logspace(-2, 4, 10)

tr_si, tr_ri = _sorted(tvsd_train_feat_idx_A)
te_si, te_ri = _sorted(tvsd_test_feat_idx_A)

# ── 4. Load neural data ──────────────────────────────────────

with h5py.File(TVSD_PATH, "r") as f:
    Y_tr = f[f"train/neural_data/{BENCH_MONKEY}/{BENCH_ROI}"][:].astype(np.float32)
    Y_te = f[f"test/neural_data/{BENCH_MONKEY}/{BENCH_ROI}"][:].astype(np.float32)

nc_ev = load_tvsd_nc(BENCH_MONKEY, BENCH_ROI)

# ── 5. Benchmark loop ────────────────────────────────

bench_results = []

with h5py.File(FEAT_A_THINGS, "r") as ff:
    bench_layers = [k for k in ff.keys() if k != "ids"][:N_LAYERS]

    for lk in bench_layers:
        X_tr = load_feat(ff, lk, tr_si, tr_ri)
        X_te = load_feat(ff, lk, te_si, te_ri)

        for n_pca in PCA_LIST:
            Y_pred = fit_ridge_encoding(X_tr, Y_tr, X_te, n_components=n_pca)

            r_raw  = np.nanmean(pearsonr_cols(Y_te, Y_pred))
            ev_raw = np.nanmean(ev_cols(Y_te, Y_pred))
            r_nc   = np.nanmean(nc_correct_r(pearsonr_cols(Y_te, Y_pred), nc_ev))
            ev_nc  = np.nanmean(nc_correct_ev(ev_cols(Y_te, Y_pred), nc_ev))

            bench_results.append({
                "layer": lk,
                "n_pca": n_pca,
                "pearsonr": r_raw,
                "ev": ev_raw,
                "pearsonr_nc": r_nc,
                "ev_nc": ev_nc,
            })

            del Y_pred

        del X_tr, X_te
        gc.collect()

del Y_tr, Y_te

bench_df = pd.DataFrame(bench_results)
print(bench_df.to_string(index=False))

# ── 6. Plot ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for layer, grp in bench_df.groupby("layer"):
    grp = grp.sort_values("n_pca")
    axes[0].plot(grp["n_pca"], grp["pearsonr_nc"], marker="o", label=layer)
    axes[1].plot(grp["n_pca"], grp["ev_nc"], marker="o", label=layer)

for ax, ylabel in zip(axes, ["Pearson r (NC)", "EV (NC)"]):
    ax.set_xscale("log", base=2)
    ax.set_xticks(PCA_LIST)
    ax.set_xticklabels(PCA_LIST)
    ax.set_xlabel("n_PCA components")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7, title="layer")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_title(f"Pearson r NC — TVSD {BENCH_MONKEY}/{BENCH_ROI}")
axes[1].set_title(f"EV NC — TVSD {BENCH_MONKEY}/{BENCH_ROI}")

plt.suptitle("PCA dimension benchmark (ResNet152-adv, first 5 layers)", fontsize=11)
plt.tight_layout()
plt.savefig("pca_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

We evaluated PCA dimensionality from 128 to 2048 components. Encoding performance peaked at 256–512 components and degraded beyond this range, despite increasing explained variance. We therefore selected 512 components as a trade-off between predictive performance and computational efficiency.

In [16]:
# ============================================================
# 2.4  Linear encoding models (ridge regression)
# ============================================================

import gc
import h5py
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# ────────────────────────────────────────────────────────────
# HDF5 SAFE ACCESS
# ────────────────────────────────────────────────────────────
def get_dataset(f, key):
    obj = f[key]

    if isinstance(obj, h5py.Dataset):
        return obj

    elif isinstance(obj, h5py.Group):
        # Try common dataset names first
        for name in ["features", "data", "activations"]:
            if name in obj:
                return obj[name]

        # fallback: first dataset inside group
        for v in obj.values():
            if isinstance(v, h5py.Dataset):
                return v

        raise ValueError(f"No dataset found inside group '{key}'")

    else:
        raise TypeError(f"Unsupported object type: {type(obj)}")

# ────────────────────────────────────────────────────────────
# Layer key extraction
# ────────────────────────────────────────────────────────────
def get_layer_keys(f):
    keys = []
    for k in f.keys():
        if k == "ids":
            continue
        obj = f[k]

        if isinstance(obj, h5py.Dataset):
            keys.append(k)

        elif isinstance(obj, h5py.Group):
            for subk in obj.keys():
                keys.append(f"{k}/{subk}")

    return sorted(keys)

# ── EV noise ceiling loading helpers ─────────────────────────────────────────
def load_tvsd_nc(monkey, roi):
    with h5py.File(TVSD_PATH, "r") as f:
        return f[f"noise_ceilings/{monkey}/{roi}"][:] / 100.0

def load_eeg_nc(subj, roi):
    with h5py.File(EEG_PATH, "r") as f:
        return f[f"noise_ceilings/{subj}/{roi}"][:] / 100.0  # (n_ch, n_tp)

def load_nsd_nc(subj, roi):
    with h5py.File(NSD_PATH, "r") as f:
        return f[f"noise_ceilings/{subj}/{roi}"][:] / 100.0

# ────────────────────────────────────────────────────────────
# Feature loading
# ────────────────────────────────────────────────────────────
def load_feat(f, layer_key, idx_sorted, idx_restore):
    dset = get_dataset(f, layer_key)
    data = dset[idx_sorted]
    return data[idx_restore].astype(np.float32)

def _sorted(idx):
    o = np.argsort(idx)
    return idx[o], np.argsort(o)

# ────────────────────────────────────────────────────────────
# Ridge model
# ────────────────────────────────────────────────────────────
N_COMP_PCA = 512
ALPHA_GRID = np.logspace(-2, 4, 10)

def fit_ridge_encoding(X_train, Y_train, X_test):

    X_train = X_train.astype(np.float32)
    X_test  = X_test.astype(np.float32)
    Y_train = Y_train.astype(np.float32)

    n_comp = min(N_COMP_PCA, X_train.shape[0]-1, X_train.shape[1])

    pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=0)
    Xtr_p = pca.fit_transform(X_train)
    Xte_p = pca.transform(X_test)

    del X_train, X_test

    sc = StandardScaler()
    Xtr_p = sc.fit_transform(Xtr_p)
    Xte_p = sc.transform(Xte_p)

    ridge = RidgeCV(alphas=ALPHA_GRID, cv=5, scoring="r2")
    ridge.fit(Xtr_p, Y_train)

    Y_pred = ridge.predict(Xte_p).astype(np.float32)

    del Xtr_p, Xte_p, Y_train
    return Y_pred

# ────────────────────────────────────────────────────────────
# Metrics
# ────────────────────────────────────────────────────────────
def pearsonr_cols(Y_true, Y_pred):
    yt = Y_true - Y_true.mean(axis=0, keepdims=True)
    yp = Y_pred - Y_pred.mean(axis=0, keepdims=True)
    num = (yt * yp).sum(axis=0)
    den = np.sqrt((yt**2).sum(axis=0) * (yp**2).sum(axis=0))
    den = np.where(den < 1e-12, np.nan, den)
    return num / den

def ev_cols(Y_true, Y_pred):
    ss_res = ((Y_true - Y_pred)**2).sum(axis=0)
    ss_tot = ((Y_true - Y_true.mean(axis=0))**2).sum(axis=0)
    ss_tot = np.where(ss_tot < 1e-12, np.nan, ss_tot)
    return 1.0 - ss_res / ss_tot

def nc_correct_r(r, nc):
    return r / np.sqrt(np.clip(nc, 1e-6, None))

def nc_correct_ev(ev, nc):
    return ev / np.clip(nc, 1e-6, None)

# ════════════════════════════════════════════════════════════
# RSA / CKA instances
# ════════════════════════════════════════════════════════════

rsa_enc = RepresentationalSimilarityAnalysis()
cka_enc = CenteredKernelAlignment()

def enc_rsa_cka(Y_true, Y_pred, n_feat=500):
    """Encoding-RSA and encoding-CKA on subsampled columns."""
    rng = np.random.default_rng(42)
    m = min(n_feat, Y_true.shape[1])
    idx = rng.choice(Y_true.shape[1], m, replace=False)
    return (rsa_enc(Y_true[:, idx], Y_pred[:, idx]),
            cka_enc(Y_true[:, idx], Y_pred[:, idx]))

# ════════════════════════════════════════════════════════════
# TVSD
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("Fitting linear encoding models")
print("=" * 60)

all_results = []

print("\n--- TVSD ---")


tvsd_tr_si_A, tvsd_tr_ri_A = _sorted(tvsd_train_feat_idx_A)
tvsd_tr_si_B, tvsd_tr_ri_B = _sorted(tvsd_train_feat_idx_B)

with h5py.File(TVSD_PATH, "r") as fneural, \
     h5py.File(FEAT_A_THINGS, "r") as fA, \
     h5py.File(FEAT_B_THINGS, "r") as fB:

    layers_A = get_layer_keys(fA)
    layers_B = get_layer_keys(fB)

    for monkey in TVSD_MONKEYS:
        for roi in TVSD_ROIS:

            Y_tr = fneural[f"train/neural_data/{monkey}/{roi}"][:].astype(np.float32)
            Y_te = fneural[f"test/neural_data/{monkey}/{roi}"][:].astype(np.float32)
            nc_ev = load_tvsd_nc(monkey, roi)

            for model_label, ffeat, layers, tr_si, tr_ri, te_si, te_ri in [
                ("ResNet152-adv", fA, layers_A,
                 tvsd_tr_si_A, tvsd_tr_ri_A, tvsd_test_si_A, tvsd_test_ri_A),
                ("Qwen3-VL-2B",   fB, layers_B,
                 tvsd_tr_si_B, tvsd_tr_ri_B, tvsd_test_si_B, tvsd_test_ri_B),
            ]:

                for li, lk in tqdm(enumerate(layers),
                                   total=len(layers),
                                   desc=f"{monkey}/{roi} {model_label}",
                                   leave=False):

                    X_tr = load_feat(ffeat, lk, tr_si, tr_ri)
                    X_te = load_feat(ffeat, lk, te_si, te_ri)

                    Y_pred = fit_ridge_encoding(X_tr, Y_tr, X_te)

                    r = np.nanmean(pearsonr_cols(Y_te, Y_pred))
                    ev = np.nanmean(ev_cols(Y_te, Y_pred))
                    r_nc = np.nanmean(nc_correct_r(pearsonr_cols(Y_te, Y_pred), nc_ev))
                    ev_nc = np.nanmean(nc_correct_ev(ev_cols(Y_te, Y_pred), nc_ev))
                    enc_rsa, enc_cka = enc_rsa_cka(Y_te, Y_pred)


                    all_results.append({
                        "dataset": "TVSD",
                        "target": f"{monkey}/{roi}",
                        "model": model_label,
                        "layer_idx": li,
                        "layer": lk,
                        "pearsonr": r,
                        "pearsonr_nc": r_nc,
                        "ev": ev,
                        "ev_nc": ev_nc,
                        "enc_rsa": enc_rsa,
                        "enc_cka": enc_cka,
                    })

                    del X_tr, X_te, Y_pred

                gc.collect()

            del Y_tr, Y_te
        gc.collect()

results_df = pd.DataFrame(all_results)
#results_df.to_csv("TVSD.csv", index=False)

print("\nDone:", len(results_df))

NameError: name 'RepresentationalSimilarityAnalysis' is not defined

In [ ]:
# ════════════════════════════════════════════════════════════
# EEG
# ════════════════════════════════════════════════════════════
print("\n--- EEG ---")

eeg_tr_si_A, eeg_tr_ri_A = _sorted(eeg_train_feat_idx_A)
eeg_tr_si_B, eeg_tr_ri_B = _sorted(eeg_train_feat_idx_B)

nc_eeg = load_eeg_nc(EEG_SUBJ, EEG_ROI)   # (n_ch, n_tp)
nc_eeg_flat = nc_eeg.ravel()

with h5py.File(EEG_PATH, "r") as fneural, \
     h5py.File(FEAT_A_THINGS, "r") as fA, \
     h5py.File(FEAT_B_THINGS, "r") as fB:

    # Load EEG
    Y_eeg_tr = fneural[f"train/neural_data/{EEG_SUBJ}/{EEG_ROI}"][:]
    Y_eeg_te = fneural[f"test/neural_data/{EEG_SUBJ}/{EEG_ROI}"][:]

    # Flatten (n_stim, n_ch * n_tp)
    n_tr, n_ch, n_tp = Y_eeg_tr.shape
    n_te = Y_eeg_te.shape[0]

    Y_tr = Y_eeg_tr.reshape(n_tr, -1).astype(np.float32)
    Y_te = Y_eeg_te.reshape(n_te, -1).astype(np.float32)

    del Y_eeg_tr, Y_eeg_te

    layers_A = get_layer_keys(fA)
    layers_B = get_layer_keys(fB)

    for model_label, ffeat, layers, tr_si, tr_ri, te_si, te_ri in [
        ("ResNet152-adv", fA, layers_A,
         eeg_tr_si_A, eeg_tr_ri_A, eeg_test_si_A, eeg_test_ri_A),
        ("Qwen3-VL-2B",   fB, layers_B,
         eeg_tr_si_B, eeg_tr_ri_B, eeg_test_si_B, eeg_test_ri_B),
    ]:

        for li, lk in tqdm(enumerate(layers),
                           total=len(layers),
                           desc=f"EEG {model_label}",
                           leave=False):

            X_tr = load_feat(ffeat, lk, tr_si, tr_ri)
            X_te = load_feat(ffeat, lk, te_si, te_ri)

            Y_pred = fit_ridge_encoding(X_tr, Y_tr, X_te)

            r_per  = pearsonr_cols(Y_te, Y_pred)
            ev_per = ev_cols(Y_te, Y_pred)

            # reliability mask
            mask = nc_eeg_flat >= 0.1

            r     = np.nanmean(r_per[mask])
            ev    = np.nanmean(ev_per[mask])
            r_nc  = np.nanmean(nc_correct_r(r_per[mask], nc_eeg_flat[mask]))
            ev_nc = np.nanmean(nc_correct_ev(ev_per[mask], nc_eeg_flat[mask]))
            enc_rsa, enc_cka = enc_rsa_cka(Y_te, Y_pred)#, n_feat=300)


            all_results.append({
                "dataset": "EEG",
                "target": f"{EEG_SUBJ}/{EEG_ROI}",
                "model": model_label,
                "layer_idx": li,
                "layer": lk,
                "pearsonr": r,
                "pearsonr_nc": r_nc,
                "ev": ev,
                "ev_nc": ev_nc,
                "enc_rsa": enc_rsa,
                "enc_cka": enc_cka,
            })

            del X_tr, X_te, Y_pred

        gc.collect()

del Y_tr, Y_te

results_df = pd.DataFrame(all_results)
#results_df.to_csv("TVSD_EEG.csv", index=False)

print("\nDone:", len(results_df))

In [ ]:
# ════════════════════════════════════════════════════════════
# NSD
# ════════════════════════════════════════════════════════════
print("\n--- NSD ---")

nsd_tr_si_A, nsd_tr_ri_A = _sorted(nsd_train_feat_idx_A[NSD_SUBJ])
nsd_tr_si_B, nsd_tr_ri_B = _sorted(nsd_train_feat_idx_B[NSD_SUBJ])
nsd_te_si_A, nsd_te_ri_A = _sorted(nsd_test_feat_idx_A[NSD_SUBJ])
nsd_te_si_B, nsd_te_ri_B = _sorted(nsd_test_feat_idx_B[NSD_SUBJ])

with h5py.File(NSD_PATH, "r") as fneural, \
     h5py.File(FEAT_A_NSD, "r") as fA, \
     h5py.File(FEAT_B_NSD, "r") as fB:

    layers_A = get_layer_keys(fA)
    layers_B = get_layer_keys(fB)

    for roi in NSD_ROIS_TARGET:

        try:
            Y_tr = fneural[f"train/neural_data/{NSD_SUBJ}/{roi}"][:].astype(np.float32)
            Y_te = fneural[f"test/neural_data/{NSD_SUBJ}/{roi}"][:].astype(np.float32)
            nc_ev = load_nsd_nc(NSD_SUBJ, roi)
        except KeyError:
            print(f"Skipping {roi}")
            continue

        for model_label, ffeat, layers, tr_si, tr_ri, te_si, te_ri in [
            ("ResNet152-adv", fA, layers_A,
             nsd_tr_si_A, nsd_tr_ri_A, nsd_te_si_A, nsd_te_ri_A),
            ("Qwen3-VL-2B",   fB, layers_B,
             nsd_tr_si_B, nsd_tr_ri_B, nsd_te_si_B, nsd_te_ri_B),
        ]:

            for li, lk in tqdm(enumerate(layers),
                               total=len(layers),
                               desc=f"NSD {roi} {model_label}",
                               leave=False):

                X_tr = load_feat(ffeat, lk, tr_si, tr_ri)
                X_te = load_feat(ffeat, lk, te_si, te_ri)

                Y_pred = fit_ridge_encoding(X_tr, Y_tr, X_te)

                r  = np.nanmean(pearsonr_cols(Y_te, Y_pred))
                ev = np.nanmean(ev_cols(Y_te, Y_pred))

                r_nc  = np.nanmean(nc_correct_r(pearsonr_cols(Y_te, Y_pred), nc_ev))
                ev_nc = np.nanmean(nc_correct_ev(ev_cols(Y_te, Y_pred), nc_ev))
                enc_rsa, enc_cka = enc_rsa_cka(Y_te, Y_pred)

                all_results.append({
                    "dataset": "NSD",
                    "target": f"{NSD_SUBJ}/{roi}",
                    "model": model_label,
                    "layer_idx": li,
                    "layer": lk,
                    "pearsonr": r,
                    "pearsonr_nc": r_nc,
                    "ev": ev,
                    "ev_nc": ev_nc,
                    "enc_rsa": enc_rsa,
                    "enc_cka": enc_cka,
                })

                del X_tr, X_te, Y_pred

            gc.collect()

        del Y_tr, Y_te
    gc.collect()

results_df = pd.DataFrame(all_results)
results_df.to_csv("results_df.csv", index=False)
print("\nDone:", len(results_df))

In [ ]:
# ============================================================
# 2.4  Encoding model plots and best-layer summary
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Layer-wise predictive performance plots ──────────────────────────────────
METRICS = ["pearsonr", "pearsonr_nc", "ev", "ev_nc", "enc_rsa", "enc_cka"]
METRIC_LABELS = {
    "pearsonr": "Pearson r", "pearsonr_nc": "Pearson r (NC)",
    "ev": "Explained Var.", "ev_nc": "EV (NC)",
    "enc_rsa": "Encoding-RSA", "enc_cka": "Encoding-CKA",
}

datasets_to_plot = {
    "TVSD":  ["monkeyF/V1","monkeyF/V4","monkeyF/IT",
              "monkeyN/V1","monkeyN/V4","monkeyN/IT"],
    "EEG":   ["sub-01/occipital_parietal"],
    "NSD":   [f"{NSD_SUBJ}/{r}" for r in NSD_ROIS_TARGET
              if f"{NSD_SUBJ}/{r}" in results_df["target"].unique()],
}

for dataset, targets in datasets_to_plot.items():
    sub_df = results_df[results_df.dataset == dataset]
    n_targets = len(targets)
    fig, axes = plt.subplots(n_targets, 2, figsize=(14, 3.5 * n_targets), squeeze=False)
    for ti, target in enumerate(targets):
        for mi, metric in enumerate(["pearsonr_nc", "enc_rsa"]):
            ax = axes[ti][mi]
            for model, ls, color in [("ResNet152-adv","--","steelblue"),
                                      ("Qwen3-VL-2B","-","tomato")]:
                row = sub_df[(sub_df.model == model) & (sub_df.target == target)]
                row = row.sort_values("layer_idx")
                ax.plot(row["layer_idx"], row[metric], color=color, ls=ls, lw=2,
                        label=model[:10])
                best_idx = row[metric].idxmax()
                ax.axvline(row.loc[best_idx, "layer_idx"], color=color, alpha=0.3, lw=1.5)
            ax.set_title(f"{target}\n{METRIC_LABELS[metric]}", fontsize=9)
            ax.set_xlabel("Layer index"); ax.set_ylabel(METRIC_LABELS[metric])
            ax.spines[["top","right"]].set_visible(False)
            if ti == 0 and mi == 0:
                ax.legend(fontsize=8)
    plt.suptitle(f"Layer-wise encoding performance — {dataset}", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"sec2_encoding_{dataset.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

# ── Best-layer summary table ─────────────────────────────────────────────────
best_rows = []
for (dataset, target, model), grp in results_df.groupby(["dataset","target","model"]):
    best = grp.loc[grp["pearsonr_nc"].idxmax()]
    best_rows.append({
        "Dataset": dataset, "Target": target, "Model": model,
        "Best layer": best["layer_idx"],
        "Pearson r": round(best["pearsonr"], 4),
        "Pearson r NC": round(best["pearsonr_nc"], 4),
        "EV": round(best["ev"], 4),
        "EV NC": round(best["ev_nc"], 4),
        "Enc-RSA": round(best["enc_rsa"], 4),
        "Enc-CKA": round(best["enc_cka"], 4),
    })

best_df = pd.DataFrame(best_rows)
print("\nBest-layer summary table:")
print(best_df.to_string(index=False))


<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.3</strong><br>Which model and which layer perform best for each dataset? Summarize the main trends in a short paragraph.</div>

For TVSD, ResNet152-adv outperforms Qwen3-VL-2B on all ROIs and both monkeys, with noise-corrected Pearson r up to 0.84 for V1 and 0.77 for IT. Its best layers tend to be early (layer 1 for V1) to mid-depth (layers 6–8 for V4 and IT), consistent with the known feedforward hierarchy of the ventral stream. For NSD, Qwen3-VL-2B achieves marginally but consistently higher scores across all ROIs (e.g., V1v: 0.84 vs 0.82; PPA: 0.82 vs 0.79), with best layers at layer 2 for high-level areas and later layers for early visual areas. For EEG, both models perform comparably (noise-corrected r ≈ 0.43), with near-zero or negative explained variance before noise correction, reflecting the difficulty of predicting broadband EEG from static image features. Overall, ResNet152-adv is stronger for macaque electrophysiology, while Qwen3-VL-2B has a slight but consistent edge in human fMRI.

---

## 2.5 Compare predictive and representational metrics

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Compare the ranking of models and layers according to:

- Pearson correlation,
- explained variance,
- RSA,
- CKA.
- encoding-RSA/ encoding-CKA

Discuss whether the same layers are favored by all metrics.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One figure comparing layer or model rankings across metrics**
2. **One concrete example where two metrics agree**
3. **One concrete example where two metrics disagree**
4. **One short written interpretation** in Answer box 2.4. (below code cells)

In [ ]:
# ============================================================
# 2.5  Compare predictive and representational metrics
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Merge encoding results with RSA/CKA results (matched by dataset, target, model, layer)
# RSA/CKA are stored in tvsd_df, eeg_time_df, nsd_rep_df

# Build a unified representational df from TVSD and NSD
rep_rows = []
for _, row in tvsd_df.iterrows():
    rep_rows.append({
        "dataset": "TVSD",
        "target": f"{row['monkey']}/{row['roi']}",
        "model": row["model"],
        "layer_idx": row["layer_idx"],
        "RSA": row["RSA"],
        "CKA": row["CKA"],
    })
for _, row in nsd_rep_df.iterrows():
    rep_rows.append({
        "dataset": "NSD",
        "target": f"{row['subj']}/{row['roi']}",
        "model": row["model"],
        "layer_idx": row["layer_idx"],
        "RSA": row["RSA"],
        "CKA": row["CKA"],
    })

# For EEG: average RSA/CKA over time points per layer
eeg_rep_agg = (eeg_time_df.groupby(["model","layer_idx"])[["RSA","CKA"]]
               .mean().reset_index())
for _, row in eeg_rep_agg.iterrows():
    rep_rows.append({
        "dataset": "EEG",
        "target": f"{EEG_SUBJ}/{EEG_ROI}",
        "model": row["model"],
        "layer_idx": int(row["layer_idx"]),
        "RSA": row["RSA"],
        "CKA": row["CKA"],
    })

rep_df_merged = pd.DataFrame(rep_rows)

# Merge with encoding results
merged = pd.merge(
    results_df[["dataset","target","model","layer_idx",
                "pearsonr_nc","ev_nc","enc_rsa","enc_cka"]],
    rep_df_merged,
    on=["dataset","target","model","layer_idx"],
    how="inner"
)

# ── Figure: ranking comparison across metrics ────────────────────────────────
# For each (dataset, target, model): normalize layer scores to [0,1] per metric,
# then plot layer index vs normalized score for each metric.

EXAMPLE_TARGETS = [
    ("TVSD", "monkeyF/IT", "ResNet152-adv"),
    ("NSD",  f"{NSD_SUBJ}/FFA-1", "Qwen3-VL-2B"),
    ("EEG",  f"{EEG_SUBJ}/{EEG_ROI}", "ResNet152-adv"),
]

metric_cols = ["pearsonr_nc", "ev_nc", "enc_rsa", "enc_cka", "RSA", "CKA"]
metric_colors = {
    "pearsonr_nc": "navy","ev_nc": "deepskyblue",
    "enc_rsa": "darkred","enc_cka": "orange",
    "RSA": "darkgreen","CKA": "limegreen",
}


fig, axes = plt.subplots(1, len(EXAMPLE_TARGETS), figsize=(16, 5), sharey=False)

for ax, (dataset, target, model) in zip(axes, EXAMPLE_TARGETS):
    sub = merged[(merged.dataset == dataset) & (merged.target == target)
                 & (merged.model == model)].sort_values("layer_idx")
    for m in metric_cols:
        vals = sub[m].values.astype(float)
        rng = vals.max() - vals.min()
        norm = (vals - vals.min()) / rng if rng > 1e-12 else vals
        ax.plot(sub["layer_idx"], norm, color=metric_colors[m], lw=2, label=m)
    ax.set_title(f"{dataset}\n{target}\n{model[:10]}", fontsize=9)
    ax.set_xlabel("Layer index")
    ax.set_ylabel("Normalized score")
    ax.legend(fontsize=7)
    ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Layer ranking comparison across metrics (normalized per metric)", fontsize=12)
plt.tight_layout()
plt.savefig("sec2_metric_ranking_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Identify agreement and disagreement ─────────────────────────────────────
print("\nMetric agreement/disagreement examples:")
print("Agreement example: TVSD monkeyF/IT, ResNet152-adv")
sub_agr = merged[(merged.dataset=="TVSD") & (merged.target=="monkeyF/IT")
                 & (merged.model=="ResNet152-adv")].sort_values("layer_idx")
print("  Best layer by pearsonr_nc:", sub_agr.loc[sub_agr["pearsonr_nc"].idxmax(), "layer_idx"])
print("  Best layer by RSA:        ", sub_agr.loc[sub_agr["RSA"].idxmax(),         "layer_idx"])
print("  Best layer by CKA:        ", sub_agr.loc[sub_agr["CKA"].idxmax(),         "layer_idx"])

nsd_target = f"{NSD_SUBJ}/V1v"
if nsd_target in merged["target"].values:
    sub_dis = merged[(merged.dataset=="NSD") & (merged.target==nsd_target)
                     & (merged.model=="Qwen3-VL-2B")].sort_values("layer_idx")
    print("\nDisagreement example: NSD V1v, Qwen3-VL-2B")
    print("  Best layer by pearsonr_nc:", sub_dis.loc[sub_dis["pearsonr_nc"].idxmax(), "layer_idx"])
    print("  Best layer by RSA:        ", sub_dis.loc[sub_dis["RSA"].idxmax(),         "layer_idx"])
    print("  Best layer by CKA:        ", sub_dis.loc[sub_dis["CKA"].idxmax(),         "layer_idx"])


<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.4</strong><br>Does a model that is representationally similar to the brain also predict neural responses well? Use at least one example from your results.</div>

Representational and predictive metrics are broadly correlated but not identical. A clear agreement example is TVSD monkeyF/IT with ResNet152-adv: pearsonr_nc, ev_nc, enc_rsa, and enc_cka all peak near the same mid-depth layers (4–6) and rise together from early layers, confirming that the layer best capturing IT's geometry also predicts individual responses well.

A clear disagreement example is the EEG case: RSA and enc_rsa diverge substantially from the predictive metrics across layers, with much noisier and inconsistent profiles. This is partly because RSA was computed per time slice while the encoding model operated on flattened responses, but it also reflects a more general point: a model layer can be geometrically similar to the brain without being highly predictive if the alignment is not well captured by a linear readout, or vice versa.

---

## 2.6 Relate layer hierarchy to brain hierarchy

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Test whether deeper layers align better with higher-level neural targets.

- Does TVSD IT align with deeper layers than V1?
- Do higher-level NSD regions prefer later layers?
- For EEG, are particular time windows associated with later layers?

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include one of the following analyses:

1. **A heatmap of layer × ROI**
2. **A ranked-layer plot by ROI**
3. **A time-resolved EEG layer comparison**

You must also include a short written conclusion in Answer box 2.5 stating whether the results support a hierarchy correspondence.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.5</strong><br>Is there evidence for a correspondence between model depth and neural hierarchy? State your conclusion clearly and support it with results.</div>

The results provide moderate support for a layer-hierarchy correspondence. In the NSD heatmap, ResNet152-adv shows the clearest gradient: early retinotopic areas (V1v–V3v) peak at layers 0–2, while higher-level areas (FFA-1, PPA, EBA) peak at layers 7–9, consistent with the hypothesis that deeper layers encode more object-level content. For Qwen3-VL-2B the gradient is weaker, with many ROIs peaking at layer 2. In TVSD, ResNet152-adv follows the expected order: V1 is best predicted by layer 1 and IT by layers 4–6, though the ordering is not strictly monotonic. For EEG, the best-aligned layer oscillates rapidly over time with no clear trend, likely reflecting low signal-to-noise in the time-resolved alignment scores rather than a genuine representational switch. Overall, the hierarchy correspondence is clearest in NSD fMRI with ResNet152-adv, and weaker or absent for the other modalities.

In [ ]:
# ============================================================
# 2.6  Layer hierarchy vs brain hierarchy
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── 1. TVSD: best layer per ROI (both models, monkeyF) ──────────────────────
tvsd_hier = []
for model in ["ResNet152-adv", "Qwen3-VL-2B"]:
    for roi in ["V1","V4","IT"]:
        sub = results_df[(results_df.dataset=="TVSD")
                         & (results_df.target==f"monkeyF/{roi}")
                         & (results_df.model==model)]
        best_layer = sub.loc[sub["pearsonr_nc"].idxmax(), "layer_idx"]
        tvsd_hier.append({"model": model, "roi": roi, "best_layer": best_layer})

tvsd_hier_df = pd.DataFrame(tvsd_hier)
print("TVSD: best layer per ROI (monkeyF)")
print(tvsd_hier_df.pivot(index="roi", columns="model", values="best_layer"))

# ── 2. NSD: layer × ROI heatmap ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ROI order from early to high-level
roi_order = ["V1v","V2v","V3v","hV4","FFA-1","VWFA-1","PPA","OPA","EBA"]
roi_order = [r for r in roi_order if f"{NSD_SUBJ}/{r}" in results_df["target"].unique()]

for ax, model in zip(axes, ["ResNet152-adv", "Qwen3-VL-2B"]):
    n_layers = results_df["layer_idx"].max() + 1
    heat = np.full((len(roi_order), n_layers), np.nan)
    for ri, roi in enumerate(roi_order):
        sub = results_df[(results_df.dataset=="NSD")
                         & (results_df.target==f"{NSD_SUBJ}/{roi}")
                         & (results_df.model==model)].sort_values("layer_idx")
        vals = sub["pearsonr_nc"].values
        if len(vals) == n_layers:
            heat[ri, :] = vals
    im = ax.imshow(heat, aspect="auto", cmap="viridis",
                   extent=[-0.5, n_layers-0.5, len(roi_order)-0.5, -0.5])
    plt.colorbar(im, ax=ax, label="Pearson r NC")
    ax.set_yticks(range(len(roi_order))); ax.set_yticklabels(roi_order, fontsize=9)
    ax.set_xlabel("Layer index"); ax.set_ylabel("NSD ROI (early → high-level)")
    ax.set_title(f"NSD layer × ROI heatmap\n{model}")

plt.suptitle("Layer hierarchy vs brain hierarchy — NSD (subj01)", fontsize=12)
plt.tight_layout()
plt.savefig("sec2_hierarchy_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 3. EEG: time-resolved best layer ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
for model, color, ls in [("ResNet152-adv","steelblue","--"),
                          ("Qwen3-VL-2B","tomato","-")]:
    sub = results_df[(results_df.dataset=="EEG") & (results_df.model==model)]
    # Note: for EEG we have one aggregated score per layer — not time-resolved from encoding
    # Use eeg_time_df for time-resolved best layer
    sub_t = eeg_time_df[eeg_time_df.model == model]
    best_per_time = sub_t.groupby("time")["RSA"].idxmax().apply(
        lambda i: sub_t.loc[i, "layer_idx"])
    ax.plot(sub_t["time"].unique(), best_per_time.values,
            color=color, ls=ls, lw=2, label=model[:10])
ax.set_xlabel("Time (s)"); ax.set_ylabel("Best layer index")
ax.set_title("EEG: best-aligned layer over time")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("sec2_eeg_best_layer_time.png", dpi=150, bbox_inches="tight")
plt.show()


---

## 2.7 Compare the two feature extractors

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Compare **Qwen3-VL-2B-Instruct** and **Adv-ResNet152** across datasets, ROIs, layers, and metrics.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One summary figure comparing Qwen3-VL-2B-Instruct and Adv-ResNet152**
2. **One table of best scores across datasets and targets**
3. **One short written interpretation** in Answer box 2.6. (below code cell)

In [ ]:
# ============================================================
# 2.7  Compare ResNet152-adv and Qwen3-VL-2B
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Best-score-per-target comparison ────────────────────────────────────────
best_per_target = []
for (dataset, target), grp in results_df.groupby(["dataset","target"]):
    for model in grp["model"].unique():
        sub = grp[grp.model == model]
        best = sub.loc[sub["pearsonr_nc"].idxmax()]
        best_per_target.append({
            "dataset": dataset, "target": target, "model": model,
            "best_pearsonr_nc": best["pearsonr_nc"],
            "best_ev_nc":       best["ev_nc"],
            "best_enc_rsa":     best["enc_rsa"],
            "best_layer":       best["layer_idx"],
        })

bpt_df = pd.DataFrame(best_per_target)

# ── Summary table ────────────────────────────────────────────────────────────
pivot_r = bpt_df.pivot_table(index=["dataset","target"],
                              columns="model",
                              values="best_pearsonr_nc")
pivot_r["advantage"] = pivot_r["Qwen3-VL-2B"] - pivot_r["ResNet152-adv"]
print("\nBest noise-corrected Pearson r per target × model:")
print(pivot_r.round(4).to_string())

# ── Figure: scatter ResNet vs Qwen across targets ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metric_pairs = [("best_pearsonr_nc","Pearson r NC"),
                ("best_ev_nc","EV NC"),
                ("best_enc_rsa","Encoding-RSA")]

colors_ds = {"TVSD":"goldenrod","EEG":"steelblue","NSD":"tomato"}

for ax, (mc, ml) in zip(axes, metric_pairs):
    pivot = bpt_df.pivot_table(index=["dataset","target"],
                                columns="model", values=mc)
    for ds, grp in pivot.groupby(level="dataset"):
        x = grp["ResNet152-adv"].values
        y = grp["Qwen3-VL-2B"].values
        ax.scatter(x, y, color=colors_ds.get(ds,"gray"), s=60, alpha=0.8, label=ds)
    lim = [min(pivot.min().min(), -0.02),
           max(pivot.max().max(), 0.02)]
    ax.plot(lim, lim, "k--", lw=1, label="y=x")
    ax.set_xlabel("ResNet152-adv"); ax.set_ylabel("Qwen3-VL-2B")
    ax.set_title(ml); ax.legend(fontsize=7)
    ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Model comparison: Qwen3-VL-2B vs Adv-ResNet152 (best layer per target)",
             fontsize=12)
plt.tight_layout()
plt.savefig("sec2_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.6</strong><br>Does the vision-language model provide a clear advantage over the CNN? Is that advantage consistent across modalities and targets?</div>

Qwen3-VL-2B's advantage over ResNet152-adv is real but modest and modality-dependent. For NSD, Qwen3-VL-2B consistently outperforms ResNet152-adv across all ROIs (advantages from +0.01 for hV4 to +0.05 for EBA and VWFA-1), with the largest gains in higher-level, semantically selective areas — plausibly reflecting richer semantic representations learned via language supervision. For EEG, the difference is negligible (+0.01). For TVSD, the pattern reverses: ResNet152-adv outperforms Qwen3-VL-2B on every ROI in both monkeys, most strongly in IT (−0.05 for monkeyF). This suggests that the adversarially trained CNN better captures the feedforward, image-computable representations of the macaque ventral stream, while the vision-language model's advantage is specific to human fMRI where semantic content plays a larger role.

---

# 3. Open-Ended Research

So far you have explored a simple encoding model with a linear readout from a single layer per subject/ROI. In this section, you will extend the baseline pipeline in one clearly defined direction. The goal is to explore a meaningful extension that goes beyond the standard linear readout and to evaluate whether it provides a practically meaningful improvement. Depth is more important than breadth: a focused experiment is better than a broad but shallow exploration.

Possible directions include:

- readouts shared across ROIs,
- readouts shared across subjects,
- readouts shared across modalities,
- combining multiple layers,
- low-rank readouts,
- nonlinear readouts,
- temporal readouts for EEG,
- attention-based readouts,
- cross-subject pooling.

## What you must include

1. **Question**  
   What are you testing?

2. **Motivation**  
   Why is this extension interesting?

3. **Method**  
   What did you change relative to the linear baseline?

4. **Comparison**  
   How does it compare to the baseline?

5. **Interpretation**  
   Did it help, and why might that be?

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **A clearly stated hypothesis**
2. **A short motivation for the extension**
3. **A clear description of the new method**
4. **One direct comparison against the linear baseline**
5. **At least one figure or one table summarizing the comparison**
6. **A short discussion of whether the extension helped in a practically meaningful way**

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 3</strong><br>State your hypothesis, summarize your result, and say whether the new method helped in a practically meaningful way.</div>

## Hypothesis

The baseline encoding pipeline uses a single model layer per ROI. We
hypothesise that combining features from multiple ResNet-152 layers will
improve brain--model alignment over the best single layer, and that the
benefit will be **largest for IT** (where no single layer dominates) and
**smallest for V1** (already well explained by one early layer), reflecting
the known increase in representational complexity along the ventral stream.

## Method

We swept 6 multi-layer subsets (Early-2 [0,1], Mid-2 [4,5], Late-2 [8,9],
Consec-3 [3,4,5], Skip-3 [0,4,9], All-10 [0–9]) against the per-ROI
best single-layer baseline on TVSD monkeyF (V1, V4, IT) using
ResNet152-adv. Each of the 10 layers was independently compressed to
$k=128$ PCA dimensions before concatenation. The same RidgeCV pipeline
(5-fold CV) was used throughout. We also ran an exhaustive sweep of all
$\binom{10}{2}=45$ two-layer pairs per ROI and visualised
$\Delta r_\mathrm{NC}$ as a heatmap.

## Summary of Results

| Subset | V1 $r_\mathrm{NC}$ | V4 $r_\mathrm{NC}$ | IT $r_\mathrm{NC}$ | Mean |
|---|---|---|---|---|
| Best-1 (baseline) | 0.752 | 0.709 | 0.742 | 0.735 |
| Early-2 [0,1]     | 0.767 | 0.625 | 0.560 | 0.651 |
| Mid-2 [4,5]       | 0.630 | 0.688 | 0.749 | 0.689 |
| Late-2 [8,9]      | 0.690 | 0.716 | 0.766 | 0.724 |
| Consec-3 [3,4,5]  | 0.664 | 0.716 | 0.758 | 0.713 |
| Skip-3 [0,4,9]    | 0.745 | 0.721 | 0.777 | 0.748 |
| All-10            | 0.767 | 0.762 | 0.791 | 0.773 |

The gain from multi-layer readouts is strongly ROI-dependent. For **IT**,
any combination that includes late layers improves over baseline (up to
$+0.049$ for All-10). For **V4**, only subsets with late layers help; early
layers actively hurt ($-0.085$ for Early-2). For **V1**, only Early-2 and
All-10 give modest gains ($+0.015$, $+0.014$); any subset dominated by
mid/late layers hurts substantially (Mid-2: $-0.122$).

Skip-3 [0,4,9] recovers most of the All-10 gain at one-third the input
dimensionality, suggesting that **hierarchical diversity** (spanning early,
mid, and late) is the key driver, not simply adding more layers.

The exhaustive 2-layer heatmap confirms this: the largest
$\Delta r_\mathrm{NC}$ values appear off-diagonal (early–late pairs), while
adjacent-layer pairs near the diagonal rarely improve over the baseline.

## Did the Extension Help?

Yes, but selectively. All-10 is the best subset overall (mean
$r_\mathrm{NC} = 0.773$ vs. baseline $0.735$, $\Delta = +0.038$), with the
largest absolute gains in IT ($+0.049$) and V4 ($+0.052$). The gain for V1
is small ($+0.014$), consistent with V1 being dominated by a single early
feature type.

The improvement is **practically meaningful for IT and V4** but modest for
V1. A key limitation is that all layers are compressed to the same rank
($k=128$), which does not account for differing intrinsic dimensionalities.
Future work could learn per-layer compression weights jointly with the
readout, or test whether the same pattern transfers to monkeyN or
Qwen3-VL-2B.

In [ ]:
# TODO: define extension
# TODO: implement method
# TODO: compare against linear baseline

In [22]:
def load_nsd_nc(subj, roi):
    with h5py.File(NSD_PATH, "r") as f:
        return f[f"noise_ceilings/{subj}/{roi}"][:] / 100.0
def load_tvsd_nc(monkey, roi):
    with h5py.File(TVSD_PATH, "r") as f:
        return f[f"noise_ceilings/{monkey}/{roi}"][:] / 100.0

def load_eeg_nc(subj, roi):
    with h5py.File(EEG_PATH, "r") as f:
        return f[f"noise_ceilings/{subj}/{roi}"][:] / 100.0  # (n_ch, n_tp)
results_df = pd.read_csv(r"~//NX-414//results_df.csv")
results_df

,dataset,target,model,layer_idx,layer,pearsonr,pearsonr_nc,ev,ev_nc,enc_rsa,enc_cka
0,TVSD,monkeyF/V1,ResNet152-adv,0,features/layer1-0,0.779278,0.802363,0.604636,0.639637,0.676725,0.848785
1,TVSD,monkeyF/V1,ResNet152-adv,1,features/layer2-0,0.782833,0.805800,0.609741,0.644038,0.722037,0.865867
2,TVSD,monkeyF/V1,ResNet152-adv,2,features/layer3-0,0.753769,0.775773,0.566568,0.598323,0.663206,0.832813
3,TVSD,monkeyF/V1,ResNet152-adv,3,features/layer3-10,0.701982,0.722293,0.491731,0.518795,0.564786,0.738047
4,TVSD,monkeyF/V1,ResNet152-adv,4,features/layer3-15,0.687098,0.706928,0.470591,0.496317,0.550609,0.720175
...,...,...,...,...,...,...,...,...,...,...,...
315,NSD,subj01/EBA,Qwen3-VL-2B,5,features/visual-blocks-14,0.481726,0.754928,0.253001,0.563584,0.710097,0.656797
316,NSD,subj01/EBA,Qwen3-VL-2B,6,features/visual-blocks-18,0.495734,0.777626,0.266881,0.598240,0.728093,0.686236
317,NSD,subj01/EBA,Qwen3-VL-2B,7,features/visual-blocks-2,0.319423,0.499116,0.106623,0.229756,0.332482,0.309910
318,NSD,subj01/EBA,Qwen3-VL-2B,8,features/visual-blocks-22,0.497766,0.780513,0.273528,0.606572,0.728333,0.694513


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Open-Ended Extension: Multi-Layer Feature Combination
# ══════════════════════════════════════════════════════════════════════════════

import
gc
import itertools
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

# ── paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path("/shared/NX-414/data")
FEAT_DIR  = Path("/shared/NX-414/extracted_features")

TVSD_PATH     = DATA_DIR / "tvsd.h5"
MODEL_A       = "adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0"
FEAT_A_THINGS = FEAT_DIR / MODEL_A / "things_stimuli.h5"

OUT_DIR = Path("part3_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── config ────────────────────────────────────────────────────────────────────
MONKEY      = "monkeyF"
ROIS        = ["V1", "V4", "IT"]
MODEL_LABEL = "ResNet152-adv"
N_COMP      = 128           # PCA dims *per layer* 
ALPHA_GRID  = np.logspace(-2, 4, 10)

# Layer subsets (indices into layer_keys list, filled after we read the file)
SUBSETS = {
    "best_1":   None,        # resolved per-ROI from single-layer sweep
    "early_2":  [0, 1],
    "mid_2":    [4, 5],
    "late_2":   [8, 9],
    "skip_3":   [0, 4, 9],
    "consec_3": [3, 4, 5],
    "all_10":   list(range(10)),
}

t0 = time.perf_counter()
def _ts(): return f"{time.perf_counter() - t0:7.1f}s"

print("=" * 65)
print("Section 3 — Multi-layer combination (standalone)")
print("=" * 65)


# ══════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _get_dataset(f, key):
    obj = f[key]
    if isinstance(obj, h5py.Dataset):
        return obj
    for name in ["features", "data", "activations"]:
        if name in obj:
            return obj[name]
    for v in obj.values():
        if isinstance(v, h5py.Dataset):
            return v
    raise ValueError(f"No dataset found inside group '{key}'")

def _get_layer_keys(f):
    keys = []
    for k in f.keys():
        if k == "ids":
            continue
        obj = f[k]
        if isinstance(obj, h5py.Dataset):
            keys.append(k)
        elif isinstance(obj, h5py.Group):
            for sk in obj.keys():
                keys.append(f"{k}/{sk}")
    return sorted(keys)

def _sorted_idx(idx):
    order   = np.argsort(idx)
    restore = np.argsort(order)
    return idx[order], restore

def _build_id_index(ids):
    return {id_: i for i, id_ in enumerate(ids)}

def _align_ids(source_ids, index):
    return np.array([index[x] for x in source_ids])

def _load_and_compress(layer_key, tr_si, tr_ri, te_si, te_ri, n_comp):
    """
    Load 1 layer from disk, PCA-compress immediately, discard raw data.
    Returns (Xtr_compressed, Xte_compressed) each shape (n, n_comp).
    """
    with h5py.File(FEAT_A_THINGS, "r") as ff:
        dset     = _get_dataset(ff, layer_key)
        X_tr_raw = dset[tr_si][tr_ri].astype(np.float32)
        X_te_raw = dset[te_si][te_ri].astype(np.float32)

    n   = int(min(n_comp, X_tr_raw.shape[0] - 1, X_tr_raw.shape[1]))
    pca = PCA(n_components=n, svd_solver="randomized", random_state=0)
    Xtr_p = pca.fit_transform(X_tr_raw)
    Xte_p = pca.transform(X_te_raw)

    del X_tr_raw, X_te_raw, pca
    gc.collect()
    return Xtr_p.astype(np.float32), Xte_p.astype(np.float32)

def _fit_ridge(X_tr, Y_tr, X_te):
    sc    = StandardScaler()
    X_tr  = sc.fit_transform(X_tr.astype(np.float32))
    X_te  = sc.transform(X_te.astype(np.float32))
    ridge = RidgeCV(alphas=ALPHA_GRID, cv=5, scoring="r2")
    ridge.fit(X_tr, Y_tr.astype(np.float32))
    return ridge.predict(X_te).astype(np.float32)

def _pearsonr_cols(Y_true, Y_pred):
    yt  = Y_true - Y_true.mean(0, keepdims=True)
    yp  = Y_pred - Y_pred.mean(0, keepdims=True)
    num = (yt * yp).sum(0)
    den = np.sqrt((yt**2).sum(0) * (yp**2).sum(0))
    return num / np.where(den < 1e-12, np.nan, den)

def _ev_cols(Y_true, Y_pred):
    ss_res = ((Y_true - Y_pred) ** 2).sum(0)
    ss_tot = ((Y_true - Y_true.mean(0)) ** 2).sum(0)
    return 1.0 - ss_res / np.where(ss_tot < 1e-12, np.nan, ss_tot)

def _nc_r(r, nc):   return r  / np.sqrt(np.clip(nc, 1e-6, None))
def _nc_ev(ev, nc): return ev / np.clip(nc, 1e-6, None)

def _scalar_metrics(Y_te, Y_pred, nc_ev):
    r  = _pearsonr_cols(Y_te, Y_pred)
    ev = _ev_cols(Y_te, Y_pred)
    return dict(
        pearsonr    = float(np.nanmean(r)),
        ev          = float(np.nanmean(ev)),
        pearsonr_nc = float(np.nanmean(_nc_r(r, nc_ev))),
        ev_nc       = float(np.nanmean(_nc_ev(ev, nc_ev))),
    )


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Stimulus indices
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n[{_ts()}] Building stimulus indices …")

with h5py.File(FEAT_A_THINGS, "r") as ff:
    feat_ids   = ff["ids"][:]
    layer_keys = _get_layer_keys(ff)

feat_index = _build_id_index(feat_ids)
n_layers   = len(layer_keys)
print(f"  {n_layers} layers: {layer_keys}")

with h5py.File(TVSD_PATH, "r") as fn:
    train_ids = fn["train/stimulus_ids"][:]
    test_ids  = fn["test/stimulus_ids"][:]

tr_si, tr_ri = _sorted_idx(_align_ids(train_ids, feat_index))
te_si, te_ri = _sorted_idx(_align_ids(test_ids,  feat_index))
print(f"  Train: {len(train_ids):,}   Test: {len(test_ids):,}")

# Clip subset indices to actual layer count
for name in SUBSETS:
    if SUBSETS[name] is not None:
        SUBSETS[name] = [i for i in SUBSETS[name] if i < n_layers]


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Build compressed feature cache (128 dims per layer)
# One layer loaded at a time 
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n[{_ts()}] Compressing features layer by layer ({N_COMP} dims/layer) …")

feat_c_tr = {}
feat_c_te = {}

for li, lk in enumerate(layer_keys):
    print(f"  [{li+1}/{n_layers}] {lk} …", end=" ", flush=True)
    feat_c_tr[lk], feat_c_te[lk] = _load_and_compress(
        lk, tr_si, tr_ri, te_si, te_ri, N_COMP)
    print(f"done  {feat_c_tr[lk].shape}")

total_mb = (sum(v.nbytes for v in feat_c_tr.values()) +
            sum(v.nbytes for v in feat_c_te.values())) / 1e6
print(f"[{_ts()}] Cache ready — total compressed RAM: {total_mb:.0f} MB")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Single-layer sweep (establishes best-layer baseline per ROI)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n[{_ts()}] Single-layer sweep …")

single_rows = []

with h5py.File(TVSD_PATH, "r") as fn:
    for roi in ROIS:
        Y_tr  = fn[f"train/neural_data/{MONKEY}/{roi}"][:].astype(np.float32)
        Y_te  = fn[f"test/neural_data/{MONKEY}/{roi}"][:].astype(np.float32)
        nc_ev = fn[f"noise_ceilings/{MONKEY}/{roi}"][:] / 100.0

        for li, lk in enumerate(layer_keys):
            Y_pred = _fit_ridge(feat_c_tr[lk], Y_tr, feat_c_te[lk])
            m = _scalar_metrics(Y_te, Y_pred, nc_ev)
            single_rows.append({"roi": roi, "layer_idx": li, "layer": lk, **m})

        best = max(
            [r for r in single_rows if r["roi"] == roi],
            key=lambda x: x["pearsonr_nc"])
        print(f"  {MONKEY}/{roi}: best layer = {best['layer_idx']}  "
              f"r_NC = {best['pearsonr_nc']:.4f}")
        del Y_tr, Y_te
        gc.collect()

single_df  = pd.DataFrame(single_rows)
best_layer = {
    roi: single_df[single_df["roi"] == roi].loc[
        single_df[single_df["roi"] == roi]["pearsonr_nc"].idxmax(), "layer"]
    for roi in ROIS
}


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Multi-layer subset sweep
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n[{_ts()}] Multi-layer subset sweep …")

subset_rows = []

with h5py.File(TVSD_PATH, "r") as fn:
    for roi in ROIS:
        Y_tr  = fn[f"train/neural_data/{MONKEY}/{roi}"][:].astype(np.float32)
        Y_te  = fn[f"test/neural_data/{MONKEY}/{roi}"][:].astype(np.float32)
        nc_ev = fn[f"noise_ceilings/{MONKEY}/{roi}"][:] / 100.0

        # baseline: best single layer
        bl_lk = best_layer[roi]
        m_bl  = _scalar_metrics(
            Y_te, _fit_ridge(feat_c_tr[bl_lk], Y_tr, feat_c_te[bl_lk]), nc_ev)
        subset_rows.append({"roi": roi, "subset": "best_1",
                             "n_layers": 1, **m_bl})
        print(f"  {roi}  best_1 ({bl_lk}):  r_NC = {m_bl['pearsonr_nc']:.4f}")

        for name, indices in SUBSETS.items():
            if name == "best_1" or not indices:
                continue
            lkeys    = [layer_keys[i] for i in indices]
            X_tr_cat = np.concatenate([feat_c_tr[lk] for lk in lkeys], axis=1)
            X_te_cat = np.concatenate([feat_c_te[lk] for lk in lkeys], axis=1)
            m        = _scalar_metrics(
                Y_te, _fit_ridge(X_tr_cat, Y_tr, X_te_cat), nc_ev)
            d = m["pearsonr_nc"] - m_bl["pearsonr_nc"]
            subset_rows.append({"roi": roi, "subset": name,
                                 "n_layers": len(lkeys), **m})
            print(f"  {roi}  {name:<12s} layers={indices}:  "
                  f"r_NC = {m['pearsonr_nc']:.4f}  Δ = {d:+.4f}")
            del X_tr_cat, X_te_cat

        del Y_tr, Y_te
        gc.collect()

sweep_df = pd.DataFrame(subset_rows)
print(f"\n[{_ts()}] Subset sweep complete.")

print("\n── r_NC summary table ──")
pivot = sweep_df.pivot_table(
    index="subset", columns="roi", values="pearsonr_nc")[ROIS]
pivot["mean"] = pivot.mean(axis=1)
print(pivot.sort_values("mean", ascending=False).round(4).to_string())


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — Exhaustive 2-layer heatmap
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n[{_ts()}] Exhaustive 2-layer sweep …")

pair_rows = []

with h5py.File(TVSD_PATH, "r") as fn:
    for roi in ROIS:
        Y_tr  = fn[f"train/neural_data/{MONKEY}/{roi}"][:].astype(np.float32)
        Y_te  = fn[f"test/neural_data/{MONKEY}/{roi}"][:].astype(np.float32)
        nc_ev = fn[f"noise_ceilings/{MONKEY}/{roi}"][:] / 100.0
        bl_nc = float(sweep_df[
            (sweep_df["roi"] == roi) & (sweep_df["subset"] == "best_1")
        ]["pearsonr_nc"].iloc[0])

        for i, j in itertools.combinations(range(n_layers), 2):
            lk_i, lk_j = layer_keys[i], layer_keys[j]
            X_tr_cat = np.concatenate([feat_c_tr[lk_i], feat_c_tr[lk_j]], axis=1)
            X_te_cat = np.concatenate([feat_c_te[lk_i], feat_c_te[lk_j]], axis=1)
            Y_pred   = _fit_ridge(X_tr_cat, Y_tr, X_te_cat)
            r_nc     = float(np.nanmean(
                _nc_r(_pearsonr_cols(Y_te, Y_pred), nc_ev)))
            pair_rows.append({"roi": roi, "i": i, "j": j,
                               "r_nc": r_nc, "delta": r_nc - bl_nc})
            del X_tr_cat, X_te_cat

        del Y_tr, Y_te
        gc.collect()
        print(f"  {roi}: done.")

pairs_df = pd.DataFrame(pair_rows)
print(f"[{_ts()}] Exhaustive sweep done ({len(pairs_df)} pairs × 3 ROIs).")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Bar chart: r_NC per subset per ROI
# ══════════════════════════════════════════════════════════════════════════════
subset_order  = ["best_1", "early_2", "mid_2", "late_2",
                 "skip_3", "consec_3", "all_10"]
subset_labels = {
    "best_1":   "Best-1\n(baseline)",
    "early_2":  "Early-2\n[0,1]",
    "mid_2":    "Mid-2\n[4,5]",
    "late_2":   "Late-2\n[8,9]",
    "skip_3":   "Skip-3\n[0,4,9]",
    "consec_3": "Consec-3\n[3,4,5]",
    "all_10":   "All-10",
}
palette = {"V1": "#4c78a8", "V4": "#f58518", "IT": "#54a24b"}

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, roi in zip(axes, ROIS):
    sub  = sweep_df[sweep_df["roi"] == roi].set_index("subset")
    bl   = float(sub.loc["best_1", "pearsonr_nc"])
    vals = [float(sub.loc[s, "pearsonr_nc"]) if s in sub.index else np.nan
            for s in subset_order]
    cols = ["#aaaaaa" if s == "best_1" else palette[roi] for s in subset_order]

    bars = ax.bar(range(len(subset_order)), vals, color=cols,
                  edgecolor="white", linewidth=0.6)
    ax.axhline(bl, color="#333333", linewidth=1.3, linestyle="--",
               label=f"baseline {bl:.3f}")
    for bar, val, s in zip(bars, vals, subset_order):
        if np.isnan(val) or s == "best_1":
            continue
        d = val - bl
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.001,
                f"{'+'if d>=0 else ''}{d:.3f}",
                ha="center", va="bottom", fontsize=6.5)
    ax.set_xticks(range(len(subset_order)))
    ax.set_xticklabels([subset_labels[s] for s in subset_order], fontsize=7.5)
    ax.set_title(f"{MONKEY} / {roi}", fontsize=11, fontweight="bold")
    ax.set_ylabel("Noise-corrected Pearson $r$" if roi == "V1" else "")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(axis="y", alpha=0.25)

plt.suptitle(
    f"Multi-layer combination vs best single layer  ·  "
    f"{MODEL_LABEL}  ·  TVSD {MONKEY}  ·  {N_COMP} PCA dims/layer",
    fontsize=10, y=1.02)
plt.tight_layout()
p = OUT_DIR / f"part3_bar_{MONKEY}.png"
plt.savefig(p, dpi=150, bbox_inches="tight")
print(f"\nSaved: {p}")
plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Heatmap: Δr_NC for every 2-layer pair
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, roi in zip(axes, ROIS):
    sub = pairs_df[pairs_df["roi"] == roi]
    mat = np.full((n_layers, n_layers), np.nan)
    for _, row in sub.iterrows():
        i, j = int(row["i"]), int(row["j"])
        mat[i, j] = mat[j, i] = row["delta"]
    vmax = np.nanmax(np.abs(mat))
    im   = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                     interpolation="nearest", aspect="auto")
    ax.set_title(f"{MONKEY} / {roi}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Layer j", fontsize=9)
    ax.set_ylabel("Layer i" if roi == "V1" else "", fontsize=9)
    ax.set_xticks(range(n_layers))
    ax.set_yticks(range(n_layers))
    plt.colorbar(im, ax=ax, shrink=0.8, label="Δr_NC vs best-1")

plt.suptitle(
    f"Δr_NC for all 2-layer pairs vs best single layer  ·  "
    f"{MODEL_LABEL}  ·  TVSD {MONKEY}",
    fontsize=10, y=1.02)
plt.tight_layout()
p = OUT_DIR / f"part3_heatmap_{MONKEY}.png"
plt.savefig(p, dpi=150, bbox_inches="tight")
print(f"Saved: {p}")
plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Single-layer profile (context for best-layer selection)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 3.8))
for roi in ROIS:
    sub = single_df[single_df["roi"] == roi].sort_values("layer_idx")
    ax.plot(sub["layer_idx"], sub["pearsonr_nc"],
            marker="o", lw=2, label=f"{MONKEY}/{roi}", color=palette[roi])
ax.set_xlabel("Layer index", fontsize=10)
ax.set_ylabel("Noise-corrected Pearson $r$", fontsize=10)
ax.set_title(f"Single-layer profile  ·  {MODEL_LABEL}  ·  TVSD {MONKEY}",
             fontsize=10)
ax.set_xticks(range(n_layers))
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
p = OUT_DIR / f"part3_layer_profile_{MONKEY}.png"
plt.savefig(p, dpi=150, bbox_inches="tight")
print(f"Saved: {p}")
plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# ANSWER BOX SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("ANSWER BOX 3 — summary numbers")
print("=" * 65)

print("\nBest subset per ROI:")
best_each = (sweep_df.sort_values("pearsonr_nc", ascending=False)
             .groupby("roi", as_index=False).first())
print(best_each[["roi", "subset", "n_layers",
                 "pearsonr_nc", "ev_nc"]].to_string(index=False))

print("\nBaseline (best_1) vs all_10:")
cmp = sweep_df[sweep_df["subset"].isin(["best_1", "all_10"])].pivot_table(
    index="roi", columns="subset", values="pearsonr_nc")[["best_1", "all_10"]]
cmp["delta"] = cmp["all_10"] - cmp["best_1"]
print(cmp.round(4).to_string())

print("\nBest 2-layer pair per ROI (from heatmap):")
best_pairs = (pairs_df.sort_values("delta", ascending=False)
              .groupby("roi", as_index=False).first())
print(best_pairs[["roi", "i", "j", "r_nc", "delta"]].to_string(index=False))

---

# Final Discussion

End the notebook with a short final discussion.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must address</div>

- Which dataset appeared noisiest?
- Which neural targets were most reliable?
- Which model aligned best overall?
- Which metrics were most consistent with each other?
- What was the main limitation of your analysis?
- What would you try next with more time?

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box final</strong><br>Write a concise final conclusion of 1–2 paragraphs summarizing your main findings and their limitations.</div>

EEG was the noisiest dataset by a wide margin, even reliable channels had noise ceilings below 20% and predictive scores were near zero before correction. NSD fMRI and TVSD electrophysiology were substantially more tractable, with early visual targets (V1v, V1) consistently yielding the highest noise-corrected scores. On model comparison, ResNet152-adv was the stronger predictor for macaque electrophysiology while Qwen3-VL-2B held a modest but consistent edge in human fMRI, particularly in semantically selective ROIs like FFA-1 and PPA. Among metrics, Pearson r_NC and explained variance NC were the most consistent with each other; RSA and CKA agreed on layer and model rankings but diverged on absolute magnitudes, most visibly in TVSD V1 where CKA substantially exceeded RSA for ResNet152-adv.
The main limitations are that all results are based on a single subject per dataset, the fixed PCA compression is not optimized per layer or ROI, and the EEG encoding model collapsed the time axis before fitting, discarding temporal structure. With more time, the most natural extensions would be averaging over subjects for more reliable alignment estimates, extending the multi-layer readout from Section 3 to NSD and EEG, and testing a small nonlinear readout for higher-level areas where the linear assumption is most likely to be restrictive.


---

# Report Content

Your **2-page PDF report** should tell a clear and coherent story. It does **not** need to reproduce every notebook result.

It can include:

1. **A brief dataset overview**
2. **An exploratory figure from Section 1**
3. **The EEG noise ceiling comparison**
4. **The NSD reliability visualization**
5. **One or two key brain–model alignment results from Section 2**


Rather, you should primarily focus on the open-ended extension you designed, describing:
- the motivation for your extension,
- the methods you implemented,
- the results you obtained,
- and the scientific insights you gained from it.


The report should emphasize interpretation, not just figures. Since the notebook is the main technical deliverable, the report should act as a **compressed scientific summary** of your most important findings rather than a figure dump.

---

# Detailed Grading Rubric

The project is graded out of **100 points** as follows:

- **Section 1: Inspection, Visualization, and Noise Ceiling Estimates — 20 points**
- **Section 2: Brain–Model Alignment — 20 points**
- **Section 3: Open-Ended Research — 30 points**
- **Report — 30 points**

**Section 0 is required but not graded separately.** It is treated as setup and reproducibility infrastructure for the rest of the notebook.

## Section 1 — 20 points

### 1.1 Dataset inspection — 3 points
- 1 pt: TVSD structure is correctly inspected and explained.
- 1 pt: EEG2 structure is correctly inspected and explained.
- 1 pt: NSD structure is correctly inspected and explained.

### 1.2 EEG visualization — 4 points
- 1 pt: example EEG time-course plot is present and readable.
- 1 pt: channel × time heatmap is present and readable.
- 1 pt: provided EEG noise ceiling visualization is present and readable.
- 1 pt: written interpretation identifies informative time windows or channel groups.

### 1.3 EEG noise ceiling estimation — 7 points
- 2 pts: variance-based estimator is implemented correctly.
- 2 pts: split-half estimator is implemented correctly.
- 1 pt: required summary visualizations are included.
- 1 pt: comparison to stored EEG noise ceilings is shown clearly.
- 1 pt: Answer box 1.3 interprets similarities and differences between estimators.

### 1.4 Statistical comparison of EEG noise ceilings — 3 points
- 1 pt: quantitative comparison table is present.
- 1 pt: statistical test or formal comparison is appropriate and correctly interpreted.
- 1 pt: final conclusion is clearly justified.

### 1.5 NSD reliability visualization — 3 points
- 1 pt: ncsnr is correctly converted and visualized on cortex.
- 1 pt: parcel overlay or parcel-wise summary is included.
- 1 pt: Answer box 1.5 correctly interprets reliable and unreliable regions.

## Section 2 — 20 points

### 2.1 RSA implementation — 3 points
- 1 pt: RDM computation is correct.
- 1 pt: RDM comparison is correct.
- 1 pt: implementation is used properly in later analyses.

### 2.2 Unbiased linear CKA implementation — 3 points
- 2 pts: unbiased linear CKA is implemented correctly.
- 1 pt: implementation is used properly in later analyses.

### 2.3 Representational analyses across layers, models, and targets — 4 points
- 1 pt: layer-wise RSA results are reported clearly.
- 1 pt: layer-wise CKA results are reported clearly.
- 1 pt: a model comparison is included.
- 1 pt: ROI-wise or time-resolved analysis is included and interpreted.

### 2.4 Predictive alignment with linear encoding models — 6 points
- 1 pt: required targets are selected and described correctly.
- 2 pts: train/validation/test procedure and ridge fitting are correct.
- 1 pt: required predictive metrics are reported correctly.
- 1 pt: encoding-RSA and encoding-CKA are reported correctly.
- 1 pt: best-layer summary and model comparison are included.

### 2.5 Compare predictive and representational metrics — 2 points
- 1 pt: ranking comparison figure is present and informative.
- 1 pt: agreement and disagreement between metrics are discussed clearly.

### 2.6 Layer hierarchy vs brain hierarchy — 1 point
- 1 pt: at least one hierarchy analysis is included and interpreted correctly.

### 2.7 Compare the two feature extractors — 1 point
- 1 pt: final comparison between Qwen3-VL and Adv-ResNet is clear and supported by results.

## Section 3 — 30 points

### Research question and motivation — 5 points
- 2 pts: research question is clear and focused.
- 3 pts: motivation is scientifically sensible and well connected to the baseline project.

### Method and implementation — 10 points
- 4 pts: the extension is described clearly.
- 4 pts: the method is implemented correctly.
- 2 pts: the design remains focused and technically appropriate for the project scope.

### Baseline comparison and evaluation — 10 points
- 4 pts: the comparison to the linear baseline is fair.
- 3 pts: at least one figure or table communicates the comparison clearly.
- 3 pts: evaluation supports the stated conclusion.

### Interpretation and limitations — 5 points
- 3 pts: the student explains whether the method helped in a practically meaningful way.
- 2 pts: limitations or caveats are acknowledged.

## Report — 30 points

### Structure and clarity — 6 points
- clear organization, readable flow, and concise scientific writing.

### Selection of results — 6 points
- the report focuses on the strongest and most relevant results rather than trying to include everything.

### Methodological correctness — 6 points
- metrics, comparisons, and claims are described accurately.

### Interpretation and synthesis — 6 points
- the report explains what the results mean and ties them back to the project goals.

### Figure quality and presentation — 6 points
- figures are readable, labeled, well-chosen, and integrated into the narrative.

## Important grading note

A submission that is technically correct but poorly interpreted will lose points. A submission with good intuition but missing required analyses will also lose points. The strongest submissions will be both **correct** and **scientifically well explained**.

---

# Final Checklist Before Submission

Before submitting, make sure that:

- group information is filled in,
- the notebook runs from top to bottom,
- all notebook outputs are cleared,
- figures have readable titles and labels,
- written answers are included in the answer boxes,
- the zip archive name follows the required format,
- no large unnecessary files are included.

---

# References

Use the references below when you need scientific context for the datasets, models, and analysis methods.

## Datasets

- Papale et al. (2025) — *An extensive dataset of spiking activity to reveal the syntax of the ventral stream*
- Gifford et al. (2022) — *A large and rich EEG dataset for modeling human visual object recognition*
- Allen et al. (2022) — *A massive 7T fMRI dataset to bridge cognitive neuroscience and artificial intelligence*
- van Bree, Styrnal, and Hebart (2025) — *How Much Variance Does Your Model Explain? A Clarifying Note On The Use Of Split-Half Reliability For Computing Noise Ceilings*

## Models

- Wong et al. (2020) — *Fast is better than free: Revisiting adversarial training*
- He et al. (2016) — *Deep Residual Learning for Image Recognition*
- Bai et al. (2025) — *Qwen3-VL Technical Report*

## Alignment and encoding

- Conwell et al. (2024) — *A large-scale examination of inductive biases shaping high-level visual representation in brains and machines*
- Gokce and Schrimpf (2025) — *Scaling Laws for Task-Optimized Models of the Primate Visual Ventral Stream*

Use these references selectively. You are not expected to read everything in full.